cell 1

In [1]:
from google.colab import files
print("Silakan unggah 3 file: resumes_train.jsonl, resumes_test.jsonl, dan job_roles.csv")
uploaded = files.upload()

Silakan unggah 3 file: resumes_train.jsonl, resumes_test.jsonl, dan job_roles.csv


Saving job_roles.csv to job_roles (1).csv
Saving resumes_test.jsonl to resumes_test (1).jsonl
Saving resumes_train.jsonl to resumes_train (1).jsonl


In [2]:
# ============================================================
# CELL 1 -> TRAINING JOB ROLE CLASSIFIER (LANGKAH 6 & 7)
# ============================================================
import pandas as pd
import json
import numpy as np
import joblib
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

print("Membaca dataset...")
# Fungsi untuk membaca file JSONL
def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return pd.DataFrame(data)

# Sesuaikan path ini dengan lokasi file Anda (di dalam data/raw/ jika sesuai langkah sebelumnya)
try:
    df_train = load_jsonl('data/raw/resumes_train.jsonl')
    df_test = load_jsonl('data/raw/resumes_test.jsonl')
    df_job_roles = pd.read_csv('data/raw/job_roles.csv')
except FileNotFoundError:
    # Fallback jika file ada di root folder
    df_train = load_jsonl('resumes_train.jsonl')
    df_test = load_jsonl('resumes_test.jsonl')
    df_job_roles = pd.read_csv('job_roles.csv')

# Pisahkan Fitur (X) dan Target (y)
X_train = df_train['Text'].fillna('')
y_train = df_train['Category']
X_test = df_test['Text'].fillna('')
y_test = df_test['Category']

print("Membangun Pipeline AI (TF-IDF + Logistic Regression)...")
# Menggunakan Logistic Regression karena sangat bagus untuk menghasilkan Probability Score (Confidence)
# Catatan: step classifier diberi nama 'classifier' (bukan 'clf') agar konsisten
# dengan format yang diharapkan oleh sel "MUAT MODEL" di bawah.
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])

print("Melatih model dengan data CV profesional...")
pipeline.fit(X_train, y_train)

# Evaluasi Model
y_pred = pipeline.predict(X_test)
akurasi = accuracy_score(y_test, y_pred)
print(f"✅ Training Selesai! Akurasi Model: {akurasi * 100:.2f}%")

# Simpan Model ke folder processed.
# Disimpan sebagai dict (bukan objek Pipeline mentah) supaya formatnya SAMA
# dengan yang diharapkan sel "MUAT MODEL": {"pipeline", "sklearn_version",
# "kategori", "normalisasi"}. Jika Anda melatih ulang di sini, hasilnya akan
# tetap kompatibel dengan sel-sel pemuatan model di bawah.
paket_model = {
    "pipeline": pipeline,
    "sklearn_version": sklearn.__version__,
    "kategori": sorted(pipeline.classes_.tolist()),
    "normalisasi": "lowercase_whitespace_v1",
    "input": "teks CV",
}

model_path = 'data/processed/job_role_classifier.joblib'
try:
    joblib.dump(paket_model, model_path)
    print(f"✅ Model berhasil disimpan di: {model_path}")
except:
    joblib.dump(paket_model, 'job_role_classifier.joblib')
    print("✅ Model berhasil disimpan di root folder.")


Membaca dataset...
Membangun Pipeline AI (TF-IDF + Logistic Regression)...
Melatih model dengan data CV profesional...
✅ Training Selesai! Akurasi Model: 91.25%
✅ Model berhasil disimpan di root folder.


In [3]:
# ============================================================
# CELL 2 -> THE CAREER BLUEPRINT ENGINE (LANGKAH 8 - 10)
# ============================================================
import re

def ekstrak_alasan_ai(cv_text, pipeline):
    """Mencari kata kunci (skill/pengalaman) yang paling memicu keputusan AI"""
    tfidf = pipeline.named_steps['tfidf']
    # Transform teks CV
    tfidf_matrix = tfidf.transform([cv_text])
    feature_names = np.array(tfidf.get_feature_names_out())

    # Ambil kata dengan skor TF-IDF tertinggi di CV ini
    sorted_idx = np.argsort(tfidf_matrix.toarray()[0])[::-1]
    top_keywords = [feature_names[i] for i in sorted_idx[:5] if tfidf_matrix.toarray()[0][i] > 0]
    return top_keywords

def dapatkan_skill_gap(cv_text, target_role, df_job_roles):
    """Mencari 1 skill penting dari job_roles.csv yang belum ada di CV"""
    role_data = df_job_roles[df_job_roles['Job Title'].str.lower() == target_role.lower()]
    if role_data.empty:
        return "Teknologi Cloud (seperti AWS atau Docker)" # Default fallback

    required_skills = role_data.iloc[0]['Required Skills'].split('|')
    cv_text_lower = cv_text.lower()

    missing_skills = [skill for skill in required_skills if skill.lower() not in cv_text_lower]
    return missing_skills[0] if missing_skills else "Sertifikasi Profesional Lanjutan"

def cetak_blueprint_karir(cv_text, pipeline, df_job_roles):
    # 1. Prediksi Probabilitas
    probs = pipeline.predict_proba([cv_text])[0]
    classes = pipeline.classes_

    # Ambil Top 3
    top3_idx = np.argsort(probs)[::-1][:3]
    top_roles = [(classes[i], probs[i] * 100) for i in top3_idx]

    peran_utama, skor_utama = top_roles[0]

    # 2. Ekstrak Wawasan
    kata_kunci = ekstrak_alasan_ai(cv_text, pipeline)
    skill_kunci_str = ", ".join([f"`{k.title()}`" for k in kata_kunci])

    # 3. Market Insights (Lookup ke CSV)
    market_data = df_job_roles[df_job_roles['Job Title'].str.lower() == peran_utama.lower()]
    kategori = market_data.iloc[0]['Category'] if not market_data.empty else "Technology & Enterprise"
    gaji = market_data.iloc[0]['Salary Range'] if not market_data.empty else "$80K - $150K"

    # 4. Actionable Roadmap (Skill Gap)
    skill_target = dapatkan_skill_gap(cv_text, peran_utama, df_job_roles)

    # ================== COPYWRITING EMOSIONAL ==================
    blueprint = f"""
🚀 LAPORAN BLUEPRINT KARIR ANDA
"Berhenti menebak-nebak. Data telah berbicara, dan inilah tempat di mana Anda akan bersinar."
{"="*70}

🌟 1. TAKDIR KARIR ANDA: {peran_utama.upper()}
Tingkat Keyakinan AI: {skor_utama:.1f}% (Sangat Kuat)
> "Dari ribuan CV profesional yang telah kami pelajari, jejak pengalaman dan
> keringat yang Anda tuangkan dalam CV ini sangat identik dengan para Top {peran_utama}
> di industri saat ini. Anda tidak perlu lagi ragu melamar di posisi ini,
> karena secara data, Anda sudah berada di jalur yang tepat."

🔍 2. MENGAPA KAMI BEGITU YAKIN? (DNA Profesional Anda)
Sistem kami tidak sekadar membaca kata, tapi membaca pola pikir Anda.
Inilah 'senjata rahasia' Anda yang langsung ditangkap oleh radar AI kami:
- Keahlian Emas Anda: Penguasaan Anda pada {skill_kunci_str} adalah bahasa yang
  paling dicari oleh perusahaan di posisi ini.
- Pola Pengalaman: Rekam jejak Anda menunjukkan ketahanan dalam problem solving
  dan arsitektur sistem—karakteristik utama seorang {peran_utama} sejati.

💼 3. NILAI ANDA DI MEDAN PERANG (Realita Pasar)
Jangan merendahkan nilai Anda. Saat ini, dunia teknologi sedang memburu talenta dengan profil seperti Anda.
- Sektor Paling Lapar  : {kategori}
- Apresiasi Gaji       : {gaji} / Tahun (Berdasarkan standar industri)
- Fakta Industri       : Dibandingkan kandidat acak, profil Anda sudah melewati
                         batas minimum persyaratan. Anda siap bersaing.

🛤️ 4. JALAN BERCABANG: Mengapa Fokus ke Sini?
Banyak pelamar yang membuang waktu melamar ke posisi yang salah. Sistem kami menyelamatkan Anda:
- {top_roles[1][0]} (Kecocokan {top_roles[1][1]:.1f}%)
  > Peluang bagus, tapi kurang optimal. Jangan buang energi Anda di zona yang membagi fokus Anda.
- {top_roles[2][0]} (Kecocokan {top_roles[2][1]:.1f}%)
  > Bukan medan tempur utama Anda. Mainkan kekuatan mutlak Anda, bukan kelemahan Anda.

🗺️ 5. PETA JALAN MENUJU TOP 10% (Tindakan Anda Selanjutnya)
Anda sudah kuat, tapi mari kita buat profil Anda tak terkalahkan di mata HRD besok pagi:
- Satu Skill Pengunci Target: Perusahaan besar yang mencari {peran_utama} kini mewajibkannya.
  Tambahkan wawasan tentang **{skill_target}** di waktu luang Anda bulan ini.
- Langkah Konkret Hari Ini: Gunakan *Fitur 1 (Simulasi HRD)* di CV-Analyzer.com kami
  untuk mencocokkan CV ini langsung dengan lowongan impian Anda!
{"="*70}
"""
    print(blueprint)

cell 3

In [4]:
# ============================================================
# CELL 3 -> INSTALL DEPENDENCY, UPLOAD CV & CETAK BLUEPRINT (STANDALONE)
# ============================================================
import subprocess
import sys

# Memastikan pymupdf terinstal di lingkungan Colab Anda saat ini
try:
    import pymupdf
except ImportError:
    print("Menginstal pymupdf...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
    import pymupdf

from google.colab import files
import io

print("Silakan unggah 1 file CV Anda (PDF) untuk diuji:")
unggahan_cv = files.upload()

if len(unggahan_cv) > 0:
    # Ambil nama dan isi file
    nama_file_cv = list(unggahan_cv.keys())[0]
    isi_file_cv = unggahan_cv[nama_file_cv]

    print(f"\n⚙️ Mengekstrak teks dari file: {nama_file_cv}...")

    # Ekstrak teks secara langsung
    teks_cv_upload = ""
    try:
        with pymupdf.open(stream=isi_file_cv, filetype="pdf") as doc:
            for page in doc:
                teks_cv_upload += page.get_text("text") + " "
    except Exception as e:
        print(f"⚠️ Gagal membaca PDF: {e}")

    if len(teks_cv_upload.strip()) < 50:
        print("⚠️ Teks terlalu pendek atau berupa gambar scan. Pastikan CV berisi teks yang bisa di-copy.")
    else:
        print("\n✨ MENGHASILKAN BLUEPRINT KARIR...\n")
        # Masukkan ke dalam mesin Blueprint (butuh 'pipeline' & 'df_job_roles' dari Sel 1).
        if "pipeline" not in globals() or "df_job_roles" not in globals():
            raise RuntimeError(
                "Jalankan sel training (CELL 1) terlebih dahulu agar "
                "'pipeline' dan 'df_job_roles' tersedia."
            )
        cetak_blueprint_karir(teks_cv_upload, pipeline, df_job_roles)

else:
    print("❌ Upload dibatalkan.")


Silakan unggah 1 file CV Anda (PDF) untuk diuji:


Saving CV_Johannes_Hutapea_Software_Engineer.pdf to CV_Johannes_Hutapea_Software_Engineer (1).pdf

⚙️ Mengekstrak teks dari file: CV_Johannes_Hutapea_Software_Engineer (1).pdf...

✨ MENGHASILKAN BLUEPRINT KARIR...


🚀 LAPORAN BLUEPRINT KARIR ANDA
"Berhenti menebak-nebak. Data telah berbicara, dan inilah tempat di mana Anda akan bersinar."

🌟 1. TAKDIR KARIR ANDA: PYTHON DEVELOPER
Tingkat Keyakinan AI: 23.2% (Sangat Kuat)
> "Dari ribuan CV profesional yang telah kami pelajari, jejak pengalaman dan
> keringat yang Anda tuangkan dalam CV ini sangat identik dengan para Top Python Developer
> di industri saat ini. Anda tidak perlu lagi ragu melamar di posisi ini,
> karena secara data, Anda sudah berada di jalur yang tepat."

🔍 2. MENGAPA KAMI BEGITU YAKIN? (DNA Profesional Anda)
Sistem kami tidak sekadar membaca kata, tapi membaca pola pikir Anda.
Inilah 'senjata rahasia' Anda yang langsung ditangkap oleh radar AI kami:
- Keahlian Emas Anda: Penguasaan Anda pada `Cloud`, `Js`, `Cloud Platfo

In [5]:
import re
import unicodedata

def bersihkan_teks(teks):
    """Menormalisasi unicode dan membersihkan spasi berlebih pada teks."""
    if not isinstance(teks, str):
        return ""
    teks = unicodedata.normalize("NFKC", teks)
    return re.sub(r"\s+", " ", teks).strip()

def buat_baris(teks, dokumen_id="uji", ekstensi="txt"):
    """Mengubah teks mentah menjadi struktur dictionary baris."""
    baris_mentah = teks.split("\n")
    sumber = []
    for i, baris in enumerate(baris_mentah):
        sumber.append({
            "id": f"L{i+1:04d}",
            "teks": baris,
            "lokasi": f"baris_{i+1}",
            "metode": ekstensi
        })
    return sumber

JUDUL_BAGIAN = {
    "identitas": [
        "identitas", "data diri", "informasi pribadi",
        "personal information", "personal details",
        "contact", "contact information", "kontak",
    ],
    "ringkasan": [
        "ringkasan", "profil", "tentang saya",
        "profil profesional", "ringkasan profil",
        "summary", "profile", "professional summary",
        "professional profile", "about me",
        "career objective", "objective",
    ],
    "pendidikan": [
        "pendidikan", "riwayat pendidikan",
        "education", "educational background",
        "academic background",
    ],
    "pengalaman": [
        "pengalaman", "pengalaman kerja", "pengalaman magang",
        "pengalaman kerja / magang",
        "pengalaman kerja/magang",
        "pengalaman kerja dan magang",
        "experience", "work experience",
        "professional experience", "employment history",
        "internship", "internship experience",
        "work and internship experience",
    ],
    "keterampilan": [
        "keterampilan", "keahlian", "kemampuan",
        "keahlian utama", "kompetensi", "kompetensi teknis",
        "skills", "technical skills", "hard skills",
        "soft skills", "core skills", "key skills",
        "technologies", "tools",
    ],
    "proyek": [
        "proyek", "pengalaman proyek",
        "projects", "project experience",
        "personal projects", "academic projects", "portfolio",
    ],
    "sertifikasi": [
        "sertifikasi", "sertifikat",
        "sertifikasi & pelatihan",
        "sertifikasi dan pelatihan",
        "certifications", "certificates",
        "licenses and certifications",
        "certifications and training",
    ],
    "organisasi": [
        "organisasi", "pengalaman organisasi",
        "organizations", "organizational experience",
        "volunteer experience", "volunteering",
    ],
    "bahasa": [
        "bahasa", "languages", "language skills",
    ],
    "penghargaan": [
        "penghargaan", "prestasi", "awards", "achievements",
    ],
}


def kunci_judul(teks):
    teks = bersihkan_teks(teks).casefold()
    teks = teks.strip(" \t:：-–—•|")
    # Samakan variasi spasi di sekitar / dan &.
    return re.sub(r"\s*([/&])\s*", r"\1", teks)


PETA_JUDUL = {
    kunci_judul(alias): kategori
    for kategori, aliases in JUDUL_BAGIAN.items()
    for alias in aliases
}


def kenali_judul(teks):
    key = kunci_judul(teks)

    if key in PETA_JUDUL:
        return PETA_JUDUL[key], ""

    # Mendukung format "Skills: Python, SQL".
    bagian = re.split(r"[:：]", teks, maxsplit=1)

    if len(bagian) == 2:
        key = kunci_judul(bagian[0])

        if key in PETA_JUDUL:
            return PETA_JUDUL[key], bagian[1].strip()

    return None, None


def pisahkan_bagian(baris, koreksi_judul=None):
    koreksi_judul = koreksi_judul or {}
    kategori_valid = set(JUDUL_BAGIAN) | {"unclassified"}

    id_sumber = {x["id"] for x in baris}
    id_tidak_ada = set(koreksi_judul) - id_sumber

    if id_tidak_ada:
        raise ValueError(
            f"ID koreksi tidak ditemukan: {sorted(id_tidak_ada)}"
        )

    hasil = []
    aktif = "unclassified"

    for sumber in baris:
        teks = sumber["teks"]

        if sumber["id"] in koreksi_judul:
            kategori = koreksi_judul[sumber["id"]]

            if kategori not in kategori_valid:
                raise ValueError(
                    f"Kategori tidak dikenal: {kategori}"
                )

            aktif = kategori
            hasil.append({
                **sumber,
                "bagian": aktif,
                "isi": "",
                "is_heading": True,
            })
            continue

        kategori, sisa = kenali_judul(teks)

        if kategori:
            aktif = kategori

            hasil.append({
                **sumber,
                "bagian": aktif,
                "isi": sisa,
                "is_heading": not bool(sisa),
            })
            continue

        judul_asing = (
            teks.endswith((":", "："))
            and len(teks.split()) <= 6
        )

        if judul_asing:
            aktif = "unclassified"

        hasil.append({
            **sumber,
            "bagian": aktif,
            "isi": "" if judul_asing else teks,
            "is_heading": judul_asing,
        })

    return hasil


print("Sel 3 selesai. Pengenalan judul bagian diperbarui.")

Sel 3 selesai. Pengenalan judul bagian diperbarui.


In [6]:
import re
import unicodedata

def bersihkan_teks(teks):
    """Menormalisasi unicode dan membersihkan spasi berlebih pada teks."""
    if not isinstance(teks, str):
        return ""
    teks = unicodedata.normalize("NFKC", teks)
    return re.sub(r"\s+", " ", teks).strip()

def buat_baris(teks, dokumen_id="uji", ekstensi="txt"):
    """Mengubah teks mentah menjadi struktur dictionary baris."""
    baris_mentah = teks.split("\n")
    sumber = []
    for i, baris in enumerate(baris_mentah):
        sumber.append({
            "id": f"L{i+1:04d}",
            "teks": baris,
            "lokasi": f"baris_{i+1}",
            "metode": ekstensi
        })
    return sumber

In [7]:
KAMUS_SKILL = {
    "Python": ["python"], "Java": ["java"], "JavaScript": ["javascript", "java script"],
    "TypeScript": ["typescript"], "C++": ["c++"], "C#": ["c#"], ".NET": [".net", "dotnet"],
    "PHP": ["php"], "HTML": ["html"], "CSS": ["css"], "SQL": ["sql"], "PostgreSQL": ["postgresql", "postgres"],
    "MySQL": ["mysql"], "MongoDB": ["mongodb"], "Redis": ["redis"], "React": ["react", "reactjs"],
    "Vue.js": ["vue.js", "vuejs"], "Node.js": ["node.js", "nodejs", "node js"],
    "Git": ["git"], "GitHub": ["github"], "Docker": ["docker"], "Google Cloud Platform": ["gcp", "google cloud platform"],
    "AWS": ["aws"], "Django": ["django"], "Flask": ["flask"], "FastAPI": ["fastapi"],
    "Express.js": ["express.js", "expressjs", "express js"], "Tailwind CSS": ["tailwind css", "tailwindcss"]
}

from collections import defaultdict

POLA_SKILL = {n: re.compile(r"(?<![\w+#])(?:" + "|".join(re.escape(a) for a in sorted(als, key=len, reverse=True)) + r")(?![\w+#])", flags=re.IGNORECASE) for n, als in KAMUS_SKILL.items()}
EMAIL = re.compile(r"(?<![\w.+-])[\w.+-]+@[\w-]+(?:\.[\w-]+)+", re.IGNORECASE)
URL = re.compile(r"(?:https?://|www\.|linkedin\.com/|github\.com/)[^\s<>|]+", re.IGNORECASE)
TELEPON = re.compile(r"(?<!\w)(?:\+?\d[\d ()-]{6,}\d)(?!\w)")

def ekstrak_entri_profil(baris_berbagian):
    import re
    from collections import defaultdict

    bagian_valid = {
        "pendidikan", "pengalaman", "proyek", "sertifikasi"
    }

    bulan = (
        r"(?:januari|februari|maret|april|mei|juni|juli|"
        r"agustus|september|oktober|november|desember|"
        r"january|february|march|may|june|july|august|"
        r"october|december|jan|feb|mar|apr|jun|jul|"
        r"aug|agu|sep|sept|oct|okt|nov|dec|des)"
    )

    titik_waktu = rf"(?:(?:{bulan})\.?\s+)?(?:19|20)\d{{2}}"

    pola_periode = re.compile(
        rf"(?<!\w){titik_waktu}\s*"
        rf"(?:[-–—]|sampai|hingga|to)\s*"
        rf"(?:{titik_waktu}|sekarang|saat ini|present|current)"
        rf"(?!\w)",
        re.IGNORECASE,
    )

    pola_institusi = re.compile(
        r"^(?:Universitas|University|Politeknik|Polytechnic|"
        r"Institut|Institute|Sekolah Tinggi|STMIK|"
        r"Akademi|Academy|College|SMK|SMA)\b",
        re.IGNORECASE,
    )

    pola_perusahaan = re.compile(
        r"^(?:PT\.?|CV\.?|Firma)\s+",
        re.IGNORECASE,
    )

    pola_jenjang = re.compile(
        r"(?<!\w)(?:D[1-4]|S[1-3]|Sarjana|Magister|Doktor|"
        r"Bachelor(?:'s)?|Master(?:'s)?|Ph\.?D\.?)(?!\w)",
        re.IGNORECASE,
    )

    labels = {
        "pendidikan": {
            "institusi": r"institusi|institution|kampus|universitas",
            "program_studi": r"program studi|prodi|jurusan|major",
            "jenjang": r"jenjang|degree",
            "ipk": r"ipk|gpa",
            "periode": r"periode|period|dates",
        },
        "pengalaman": {
            "perusahaan": r"perusahaan|company|employer",
            "jabatan": r"posisi|jabatan|position|job title|role",
            "periode": r"periode|period|dates",
        },
        "proyek": {
            "nama_proyek": r"nama proyek|project name",
            "peran": r"peran|posisi|role|position",
            "periode": r"periode|period|dates",
        },
        "sertifikasi": {
            "nama_sertifikasi": r"nama sertifikasi|certification name",
            "penerbit": r"penerbit|issuer",
            "tanggal": r"tanggal|date|tahun|year",
        },
    }

    field_utama = {
        "pendidikan": "institusi",
        "pengalaman": "perusahaan",
        "proyek": "nama_proyek",
        "sertifikasi": "nama_sertifikasi",
    }

    entri = []
    belum = []
    nomor = defaultdict(int)
    aktif = None

    def tepi(teks):
        return teks.strip(" \t—–-|,;")

    def baru(bagian):
        nomor[bagian] += 1
        item = {
            "id_entri": f"{bagian}_{nomor[bagian]}",
            "bagian": bagian,
            "fields": [],
            "baris": [],
            "status": "kandidat_perlu_review",
        }
        entri.append(item)
        return item

    def tambah_field(item, atribut, nilai, sumber, metode):
        nilai = nilai.strip()
        if not nilai: return
        key = (atribut, nilai, sumber["id"])
        if any((f["atribut"], f["nilai_asli"], f["baris_id"]) == key for f in item["fields"]): return
        item["fields"].append({
            "atribut": atribut,
            "nilai_asli": nilai,
            "baris_id": sumber["id"],
            "bukti": sumber["teks"],
            "metode": metode,
        })

    for sumber in baris_berbagian:
        bagian = sumber["bagian"]
        if bagian not in bagian_valid or sumber["is_heading"]:
            aktif = None; continue
        if aktif is not None and aktif["bagian"] != bagian:
            aktif = None
        teks = sumber["isi"].strip()
        if not teks: continue

        atribut_label, nilai_label = None, None
        for atribut, label in labels[bagian].items():
            match = re.match(rf"^(?:{label})\s*[:：]\s*(.+)$", teks, flags=re.IGNORECASE)
            if match:
                atribut_label = atribut
                nilai_label = match.group(1).strip()
                break

        periode = pola_periode.search(teks)
        header_fields = []
        awal_baru = atribut_label == field_utama[bagian]

        bullet = bool(re.match(r"^[•●▪*]\s*|^[-–—]\s+", teks))
        if atribut_label is None and not bullet:
            kepala = tepi(teks[:periode.start()] if periode else teks)
            pasangan = re.split(r"\s+[—–|]\s+", kepala, maxsplit=1)

            if bagian == "pendidikan" and pola_institusi.match(kepala):
                awal_baru = True
                jenjang = pola_jenjang.search(kepala)
                if len(pasangan) == 2: institusi = tepi(pasangan[0])
                elif jenjang: institusi = tepi(kepala[:jenjang.start()])
                else: institusi = kepala
                header_fields.append(("institusi", institusi))
                if jenjang:
                    header_fields.append(("jenjang", jenjang.group()))
                    program = tepi(kepala[jenjang.end():])
                    if program: header_fields.append(("program_studi", program))
            elif bagian == "pengalaman":
                if pola_perusahaan.match(kepala) or (len(pasangan) == 2 and periode):
                    awal_baru = True
                    header_fields.append(("perusahaan", tepi(pasangan[0])))
                    if len(pasangan) == 2 and tepi(pasangan[1]):
                        header_fields.append(("jabatan", tepi(pasangan[1])))
            elif bagian == "proyek" and len(pasangan) == 2 and periode:
                awal_baru = True
                header_fields.extend([("nama_proyek", tepi(pasangan[0])), ("peran", tepi(pasangan[1]))])

        if awal_baru: aktif = baru(bagian)

        if aktif is None:
            belum.append({"bagian": bagian, "baris_id": sumber["id"], "teks": teks, "bukti": sumber["teks"], "alasan": "Awal entri belum dikenali."})
            continue

        aktif["baris"].append({"baris_id": sumber["id"], "teks": teks, "bukti": sumber["teks"]})

        if atribut_label: tambah_field(aktif, atribut_label, nilai_label, sumber, "label_eksplisit")
        for atribut, nilai in header_fields: tambah_field(aktif, atribut, nilai, sumber, "pola_header")
        if periode and atribut_label not in {"periode", "tanggal"}: tambah_field(aktif, "periode", periode.group(), sumber, "pola_periode")

    return {"entri": entri, "belum_terstruktur": belum}

def ekstrak_profil(baris_berbagian):
    from collections import defaultdict  # jaga-jaga jika sel ini dijalankan ulang sendiri
    fakta, isi_bagian = [], defaultdict(list)
    for sumber in baris_berbagian:
        if sumber["is_heading"] or not sumber["isi"]: continue
        teks, bagian = sumber["isi"], sumber["bagian"]
        isi_bagian[bagian].append({"baris_id": sumber["id"], "teks": teks, "bukti": sumber["teks"]})

        for m in EMAIL.finditer(teks): fakta.append({"jenis": "email", "atribut": "email", "nilai_asli": m.group(), "nilai_normal": m.group(), "bagian": bagian, "baris_id": sumber["id"], "bukti": sumber["teks"]})
        for m in URL.finditer(teks): fakta.append({"jenis": "tautan", "atribut": "tautan", "nilai_asli": m.group(), "nilai_normal": m.group(), "bagian": bagian, "baris_id": sumber["id"], "bukti": sumber["teks"]})
        for m in TELEPON.finditer(teks): fakta.append({"jenis": "telepon", "atribut": "telepon", "nilai_asli": m.group(), "nilai_normal": m.group(), "bagian": bagian, "baris_id": sumber["id"], "bukti": sumber["teks"]})

        if bagian not in {"bahasa", "penghargaan"}:
            for nama, pola in POLA_SKILL.items():
                for m in pola.finditer(teks): fakta.append({"jenis": "skill", "atribut": nama, "nilai_asli": m.group(), "nilai_normal": nama, "bagian": bagian, "baris_id": sumber["id"], "bukti": sumber["teks"]})

    res = ekstrak_entri_profil(baris_berbagian)
    return {"fakta": fakta, "isi_bagian": dict(isi_bagian), "entri": res["entri"], "belum_terstruktur": res["belum_terstruktur"]}

print("Sel 4 selesai. Logika deteksi entri diperbarui.")

Sel 4 selesai. Logika deteksi entri diperbarui.


cell 6

In [8]:
def validasi_entri_profil(entri_list, sumber_baris):
    import re
    import unicodedata
    from collections import defaultdict

    def normal(teks):
        teks = unicodedata.normalize("NFKC", str(teks))
        return re.sub(r"\s+", " ", teks).strip()

    sumber_map = {x["id"]: x for x in sumber_baris}
    keluaran = []

    for entri in entri_list:
        fields_valid, fields_ditahan, baris_valid, catatan = [], [], [], []

        for baris in entri.get("baris", []):
            sumber = sumber_map.get(baris.get("baris_id"))
            if (sumber is None or baris.get("bukti") != sumber["teks"] or not normal(baris.get("teks", "")) or normal(baris["teks"]) not in normal(sumber["teks"])):
                catatan.append(f"Baris {baris.get('baris_id')} tidak lolos pemeriksaan bukti.")
            else: baris_valid.append(dict(baris))

        id_baris_valid = {b["baris_id"] for b in baris_valid}

        for field in entri.get("fields", []):
            sumber = sumber_map.get(field.get("baris_id"))
            nilai, alasan = normal(field.get("nilai_asli", "")), []
            if sumber is None: alasan.append("ID sumber tidak ditemukan.")
            else:
                if field.get("bukti") != sumber["teks"]: alasan.append("Bukti tidak sama dengan sumber.")
                if not nilai or nilai not in normal(sumber["teks"]): alasan.append("Nilai tidak ditemukan pada sumber.")
            if field.get("baris_id") not in id_baris_valid: alasan.append("Field tidak berasal dari baris valid entri.")

            if alasan: fields_ditahan.append({**field, "status": "ditahan", "alasan": alasan})
            else: fields_valid.append({**field, "nilai_normal": nilai, "status": "bukti_cocok_perlu_review"})

        nilai_per_field = defaultdict(set)
        for field in fields_valid: nilai_per_field[field["atribut"]].add(field["nilai_normal"])
        for atribut, nilai in nilai_per_field.items():
            if len(nilai) > 1: catatan.append(f"Konflik {atribut}: {sorted(nilai)}. Periksa batas entri.")

        if any(f.get("metode") == "pola_header" for f in fields_valid): catatan.append("Interpretasi header format bebas perlu diperiksa.")
        catatan.append("Hubungan antarbaris berdasarkan urutan dokumen; bukti cocok belum menjamin pengelompokan benar.")

        status = "perlu_perbaikan_bukti" if fields_ditahan or len(baris_valid) != len(entri.get("baris", [])) else "kandidat_perlu_review"

        keluaran.append({**entri, "baris": baris_valid, "fields": fields_valid, "fields_ditahan": fields_ditahan, "catatan": catatan, "status": status})
    return keluaran

def normalisasi_validasi(profil, sumber_baris):
    from collections import defaultdict  # jaga-jaga jika sel ini dijalankan ulang sendiri
    sumber_map = {x["id"]: x for x in sumber_baris}
    valid, ditahan, seen_fakta = [], [], set()
    skill_map = defaultdict(list)

    for fakta in profil.get("fakta", []):
        nilai = bersihkan_teks(fakta["nilai_asli"])
        sumber = sumber_map.get(fakta["baris_id"])
        if not nilai or sumber is None or nilai not in bersihkan_teks(sumber["teks"]):
            ditahan.append({**fakta, "alasan": ["Bukti tidak cocok"]}); continue

        fakta_key = (fakta["jenis"], fakta["atribut"], fakta["baris_id"])
        if fakta_key not in seen_fakta:
            valid.append(fakta)
            seen_fakta.add(fakta_key)
            if fakta["jenis"] == "skill":
                skill_map[fakta["atribut"]].append({"baris_id": fakta["baris_id"], "bagian": fakta["bagian"], "bukti": fakta["bukti"]})

    return {
        "status": "draft_perlu_review", "fakta": valid, "skill_ringkas": [{"skill": s, "bukti": b} for s, b in sorted(skill_map.items())],
        "isi_bagian": profil.get("isi_bagian", {}), "fakta_ditahan": ditahan, "peringatan": [],
        "entri": validasi_entri_profil(profil.get("entri", []), sumber_baris),
        "belum_terstruktur": profil.get("belum_terstruktur", []),
    }

def analisis_baris(baris, koreksi_judul=None):
    terpisah = pisahkan_bagian(baris, koreksi_judul=koreksi_judul)
    profil = ekstrak_profil(terpisah)
    hasil = normalisasi_validasi(profil, baris)
    return terpisah, hasil

def proses_teks_uji(teks):
    sumber = buat_baris(teks, "uji", "txt")
    terpisah, hasil_uji = analisis_baris(sumber)
    return sumber, terpisah, hasil_uji

def skill_dari(hasil_uji):
    return {item["skill"] for item in hasil_uji.get("skill_ringkas", [])}

print("Sel 5 diperbaiki. Validasi bukti entri diperketat.")

Sel 5 diperbaiki. Validasi bukti entri diperketat.


In [9]:
# ============================================================
# SEL BACA CV — MEMBACA PDF / DOCX / PNG / JPG / TXT
# ============================================================
# Fungsi ini sebelumnya dipanggil di Sel 6 tapi tidak pernah didefinisikan
# di mana pun pada notebook ini (bug utama). Sel ini menambahkannya.
#
# baca_cv(nama_file, isi_bytes, paksa_ocr=False) -> (sumber_baris, laporan)
#   - sumber_baris: daftar baris terstruktur, format sama dengan buat_baris().
#   - laporan: daftar dict berisi tahapan pembacaan (untuk ditampilkan sebagai tabel).
import io
import os
import shutil
import subprocess
import sys


def _pastikan_paket(nama_modul, nama_pip=None):
    """Import sebuah modul, meng-install paketnya dulu lewat pip jika belum ada."""
    try:
        return __import__(nama_modul)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", nama_pip or nama_modul]
        )
        return __import__(nama_modul)


def _pastikan_tesseract():
    """Pastikan biner tesseract-ocr tersedia di sistem (untuk OCR gambar/PDF scan)."""
    if shutil.which("tesseract"):
        return True
    try:
        subprocess.run(
            ["apt-get", "-qq", "update"],
            check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "tesseract-ocr"],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
    except Exception:
        return False
    return shutil.which("tesseract") is not None


def _ocr_gambar_bytes(isi_bytes, laporan, tahap_label="ocr_gambar"):
    """OCR satu gambar (bytes) menjadi teks polos. Menambah catatan ke `laporan`."""
    try:
        Image = _pastikan_paket("PIL", "Pillow").Image
        pytesseract = _pastikan_paket("pytesseract")
    except Exception as e:
        laporan.append({"tahap": tahap_label, "status": "gagal", "detail": f"Gagal memuat paket OCR: {e}"})
        return ""

    if not _pastikan_tesseract():
        laporan.append({
            "tahap": tahap_label,
            "status": "gagal",
            "detail": (
                "Biner tesseract-ocr tidak tersedia dan gagal dipasang otomatis. "
                "Jalankan '!apt-get install -y tesseract-ocr' secara manual di Colab, "
                "lalu jalankan ulang sel ini."
            ),
        })
        return ""

    try:
        gambar = Image.open(io.BytesIO(isi_bytes))
        teks = pytesseract.image_to_string(gambar, lang="eng")
        laporan.append({
            "tahap": tahap_label, "status": "ok",
            "detail": f"{len(teks)} karakter terbaca lewat OCR.",
        })
        return teks
    except Exception as e:
        laporan.append({"tahap": tahap_label, "status": "gagal", "detail": str(e)})
        return ""


def _ocr_pdf_bytes(isi_bytes, laporan):
    """Rasterisasi tiap halaman PDF lalu OCR — untuk PDF hasil scan (tanpa teks)."""
    try:
        pymupdf = _pastikan_paket("pymupdf")
    except Exception as e:
        laporan.append({"tahap": "ocr_pdf", "status": "gagal", "detail": f"Gagal memuat pymupdf: {e}"})
        return ""

    teks_total = []
    try:
        with pymupdf.open(stream=isi_bytes, filetype="pdf") as doc:
            for i, page in enumerate(doc):
                pix = page.get_pixmap(dpi=300)
                png_bytes = pix.tobytes("png")
                teks_halaman = _ocr_gambar_bytes(
                    png_bytes, laporan, tahap_label=f"ocr_pdf_halaman_{i + 1}"
                )
                teks_total.append(teks_halaman)
    except Exception as e:
        laporan.append({"tahap": "ocr_pdf", "status": "gagal", "detail": str(e)})

    return "\n".join(teks_total)


def baca_cv(nama_file, isi_bytes, paksa_ocr=False):
    """
    Membaca satu file CV (PDF, DOCX, PNG, JPG, atau TXT) dan mengembalikan
    (sumber_baris, laporan). Untuk PDF yang ternyata hasil scan (tanpa teks
    yang bisa diekstrak langsung), otomatis jatuh ke OCR. `paksa_ocr=True`
    memaksa jalur OCR meski PDF punya teks langsung.
    """
    if isinstance(isi_bytes, bytearray):
        isi_bytes = bytes(isi_bytes)

    ekstensi = os.path.splitext(nama_file)[1].lower().lstrip(".")
    laporan = []
    teks = ""
    metode = ekstensi or "txt"

    if ekstensi == "txt":
        try:
            teks = isi_bytes.decode("utf-8")
        except UnicodeDecodeError:
            teks = isi_bytes.decode("latin-1", errors="replace")
            laporan.append({
                "tahap": "baca_txt", "status": "peringatan",
                "detail": "Encoding bukan UTF-8, dibaca sebagai latin-1.",
            })
        laporan.append({"tahap": "baca_txt", "status": "ok", "detail": f"{len(teks)} karakter terbaca."})
        metode = "txt"

    elif ekstensi == "pdf":
        try:
            pymupdf = _pastikan_paket("pymupdf")
        except Exception as e:
            laporan.append({"tahap": "baca_pdf_teks", "status": "gagal", "detail": f"Gagal memuat pymupdf: {e}"})
            pymupdf = None

        teks_langsung, jumlah_halaman = "", 0
        if pymupdf is not None:
            try:
                with pymupdf.open(stream=isi_bytes, filetype="pdf") as doc:
                    jumlah_halaman = len(doc)
                    for page in doc:
                        teks_langsung += page.get_text("text") + "\n"
            except Exception as e:
                laporan.append({"tahap": "baca_pdf_teks", "status": "gagal", "detail": str(e)})

        cukup_teks = len(teks_langsung.strip()) >= 50

        if not paksa_ocr and cukup_teks:
            teks = teks_langsung
            metode = "pdf_teks"
            laporan.append({
                "tahap": "baca_pdf_teks", "status": "ok",
                "detail": f"{jumlah_halaman} halaman, {len(teks_langsung)} karakter.",
            })
        else:
            alasan = (
                "paksa_ocr=True (OCR diminta secara eksplisit)."
                if paksa_ocr else
                "Teks langsung terlalu pendek/kosong — kemungkinan PDF hasil scan."
            )
            laporan.append({"tahap": "baca_pdf_teks", "status": "dilewati", "detail": alasan})
            teks = _ocr_pdf_bytes(isi_bytes, laporan)
            metode = "pdf_ocr"

    elif ekstensi == "docx":
        try:
            docx = _pastikan_paket("docx", "python-docx")
            dokumen = docx.Document(io.BytesIO(isi_bytes))
            baris_docx = [p.text for p in dokumen.paragraphs]
            for tabel in dokumen.tables:
                for row in tabel.rows:
                    baris_docx.append(" | ".join(cell.text for cell in row.cells))
            teks = "\n".join(baris_docx)
            metode = "docx"
            laporan.append({
                "tahap": "baca_docx", "status": "ok",
                "detail": f"{len(dokumen.paragraphs)} paragraf, {len(dokumen.tables)} tabel.",
            })
        except Exception as e:
            laporan.append({"tahap": "baca_docx", "status": "gagal", "detail": str(e)})

    elif ekstensi in {"png", "jpg", "jpeg"}:
        teks = _ocr_gambar_bytes(isi_bytes, laporan, tahap_label="ocr_gambar")
        metode = "gambar_ocr"

    else:
        laporan.append({
            "tahap": "deteksi_format", "status": "gagal",
            "detail": f"Ekstensi .{ekstensi or '(tidak ada)'} tidak didukung. Gunakan PDF, DOCX, PNG, JPG, atau TXT.",
        })

    if len(teks.strip()) < 10:
        laporan.append({
            "tahap": "validasi_akhir", "status": "peringatan",
            "detail": "Teks yang berhasil dibaca sangat pendek atau kosong.",
        })

    sumber_baris = buat_baris(teks, dokumen_id=nama_file, ekstensi=metode)
    laporan.append({
        "tahap": "selesai", "status": "ok",
        "detail": f"{len(sumber_baris)} baris dihasilkan dari '{nama_file}'.",
    })
    return sumber_baris, laporan


print("Sel baca_cv siap. Mendukung PDF, DOCX, PNG, JPG, dan TXT (dengan fallback OCR untuk PDF/gambar scan).")


Sel baca_cv siap. Mendukung PDF, DOCX, PNG, JPG, dan TXT (dengan fallback OCR untuk PDF/gambar scan).


In [10]:
PAKAI_CONTOH = False
PAKSA_OCR = False

# Gunakan hanya untuk menandai baris judul yang tidak dikenali.
# Contoh: {"L0026": "pengalaman"}
KOREKSI_JUDUL = {}

fungsi_wajib = [
    "baca_cv", "pisahkan_bagian", "ekstrak_profil",
    "normalisasi_validasi", "analisis_baris",
]

belum_tersedia = [
    nama for nama in fungsi_wajib
    if not callable(globals().get(nama))
]

if belum_tersedia:
    raise RuntimeError(
        "Jalankan Sel 1–5 terlebih dahulu. Belum tersedia: "
        + ", ".join(belum_tersedia)
    )

# Kosongkan hasil lama jika proses baru gagal.
hasil = None
sumber_baris = None
baris_terpisah = None

CONTOH_CV = """Nama: Andi Pratama
Email: andi@example.com
Telepon: +62 812-3456-7890

PROFIL PROFESIONAL
Mahasiswa informatika dengan pengalaman proyek aplikasi web.

KEAHLIAN UTAMA
Python, Django, PostgreSQL, Git, Express.js, Tailwind CSS

PENGALAMAN PROYEK
Nama Proyek: Portal Laporan Warga
Membuat REST API menggunakan Django REST Framework.
Melakukan deployment menggunakan Docker dan GCP.

PENGALAMAN KERJA / MAGANG
Perusahaan: PT Contoh Digital
Posisi: Web Developer Intern
Periode: Januari 2025–April 2025
Membangun dashboard inventaris menggunakan Django.

PENDIDIKAN
Institusi: Politeknik Contoh
Program Studi: Manajemen Informatika
Jenjang: D3
Periode: 2022–2025

SERTIFIKASI & PELATIHAN
Nama Sertifikasi: Python Dasar
Penerbit: Lembaga Contoh
Tanggal: 2024
"""

if PAKAI_CONTOH:
    nama_cv = "contoh_sintetis.txt"
    isi_cv = CONTOH_CV.encode("utf-8")
else:
    print("Unggah satu CV: PDF, DOCX, PNG, JPG, atau TXT.")
    unggahan = files.upload()

    if len(unggahan) != 1:
        raise ValueError(
            "Unggah tepat satu CV, lalu jalankan ulang Sel 6."
        )

    nama_cv, isi_cv = next(iter(unggahan.items()))

sumber_baris, laporan = baca_cv(
    nama_cv,
    isi_cv,
    paksa_ocr=PAKSA_OCR,
)

baris_terpisah, hasil = analisis_baris(
    sumber_baris,
    koreksi_judul=KOREKSI_JUDUL,
)

print("\nFILE:", nama_cv)

print("\nA. LAPORAN PEMBACAAN")
display(pd.DataFrame(laporan))

print("\nB. TEKS CV YANG TERBACA")
display(pd.DataFrame(sumber_baris)[[
    "id", "lokasi", "metode", "teks"
]])

print("\nC. PEMISAHAN BAGIAN CV")
display(pd.DataFrame(baris_terpisah)[[
    "id", "bagian", "is_heading", "isi"
]])

print("\nD. FAKTA HASIL EKSTRAKSI")
if hasil["fakta"]:
    display(pd.DataFrame(hasil["fakta"])[[
        "jenis", "atribut", "nilai_asli", "nilai_normal",
        "bagian", "baris_id", "bukti",
    ]])
else:
    print("Belum ada fakta yang berhasil diekstrak.")

print("\nE. KEMUNCULAN SKILL DAN BUKTINYA")
skill_rows = [
    {
        "skill": item["skill"],
        "bagian": bukti["bagian"],
        "baris_id": bukti["baris_id"],
        "bukti": bukti["bukti"],
    }
    for item in hasil["skill_ringkas"]
    for bukti in item["bukti"]
]

if skill_rows:
    display(pd.DataFrame(skill_rows))
else:
    print("Belum ada istilah yang cocok dengan kamus skill.")

print("\nF. ISI PROFIL PER BAGIAN")
for kategori, items in hasil["isi_bagian"].items():
    print(f"\n{kategori.upper()}")
    display(pd.DataFrame(items))

print("\nG. FAKTA DITAHAN")
if hasil["fakta_ditahan"]:
    display(pd.DataFrame(hasil["fakta_ditahan"]))
else:
    print("Tidak ada.")

print("\nCATATAN REVIEW")
for pesan in hasil["peringatan"]:
    print("-", pesan)

print(
    "\nHasil tersedia pada variabel 'hasil'. "
    "Bandingkan dengan CV asli; tidak ada file hasil yang dibuat."
)

print("\nH. ENTRI PROFIL TERSTRUKTUR")

for entri in hasil.get("entri", []):
    print(
        f"\n{entri['id_entri']} "
        f"— {entri['status']}"
    )

    if entri["fields"]:
        display(pd.DataFrame(entri["fields"])[[
            "atribut",
            "nilai_asli",
            "baris_id",
            "metode",
            "status",
        ]])

    print("Baris yang terhubung ke entri:")
    display(pd.DataFrame(entri["baris"]))

    for catatan in entri["catatan"]:
        print("-", catatan)

    if entri["fields_ditahan"]:
        print("Field yang ditahan:")
        display(pd.DataFrame(entri["fields_ditahan"]))

print("\nI. BARIS YANG BELUM BISA DIBENTUK MENJADI ENTRI")

if hasil.get("belum_terstruktur"):
    display(pd.DataFrame(hasil["belum_terstruktur"]))
else:
    print("Tidak ada baris awal entri yang belum terstruktur.")

Unggah satu CV: PDF, DOCX, PNG, JPG, atau TXT.


Saving CV_Kesha_Amelia_Mobile_Developer.pdf to CV_Kesha_Amelia_Mobile_Developer.pdf

FILE: CV_Kesha_Amelia_Mobile_Developer.pdf

A. LAPORAN PEMBACAAN


,tahap,status,detail
0,baca_pdf_teks,ok,"1 halaman, 2585 karakter."
1,selesai,ok,47 baris dihasilkan dari 'CV_Kesha_Amelia_Mobi...



B. TEKS CV YANG TERBACA


,id,lokasi,metode,teks
0,L0001,baris_1,pdf_teks,KESHA AMELIA PRASETYO
1,L0002,baris_2,pdf_teks,MOBILE DEVELOPER / FLUTTER DEVELOPER
2,L0003,baris_3,pdf_teks,"Bandung, Indonesia | +62813-9876-5432 | ke..."
3,L0004,baris_4,pdf_teks,linkedin.com/in/keshaamelia | github.com/kes...
4,L0005,baris_5,pdf_teks,PROFIL PROFESIONAL
5,L0006,baris_6,pdf_teks,Lulusan Rekayasa Perangkat Lunak yang memiliki...
6,L0007,baris_7,pdf_teks,platform. Terbiasa membangun aplikasi mobile d...
7,L0008,baris_8,pdf_teks,mengintegrasikan RESTful API dan layanan berba...
8,L0009,baris_9,pdf_teks,antarmuka pengguna yang responsif dan siap ber...
9,L0010,baris_10,pdf_teks,KEAHLIAN UTAMA


...

,atribut,nilai_asli,baris_id,metode,status
0,institusi,Universitas Bina Cendekia,L0039,pola_header,bukti_cocok_perlu_review
1,jenjang,S1,L0039,pola_header,bukti_cocok_perlu_review
2,program_studi,Teknik Informatika,L0039,pola_header,bukti_cocok_perlu_review
3,periode,2022 – 2026,L0040,pola_periode,bukti_cocok_perlu_review
4,ipk,3.80/4.00 — Judul Skripsi: Implementasi Arsite...,L0041,label_eksplisit,bukti_cocok_perlu_review


Baris yang terhubung ke entri:


,baris_id,teks,bukti
0,L0039,Universitas Bina Cendekia — S1 Teknik Informatika,Universitas Bina Cendekia — S1 Teknik Informatika
1,L0040,2022 – 2026,2022 – 2026
2,L0041,IPK: 3.80/4.00 — Judul Skripsi: Implementasi A...,IPK: 3.80/4.00 — Judul Skripsi: Implementasi A...
3,L0042,Transportasi Real-Time,Transportasi Real-Time


- Interpretasi header format bebas perlu diperiksa.
- Hubungan antarbaris berdasarkan urutan dokumen; bukti cocok belum menjamin pengelompokan benar.

I. BARIS YANG BELUM BISA DIBENTUK MENJADI ENTRI


,bagian,baris_id,teks,bukti,alasan
0,proyek,L0017,"Aplikasi E-Commerce ""BelanjaYuk"" — Mobile Deve...","Aplikasi E-Commerce ""BelanjaYuk"" — Mobile Deve...",Awal entri belum dikenali.
1,proyek,L0018,April 2026 – Agustus 2026,April 2026 – Agustus 2026,Awal entri belum dikenali.
2,proyek,L0019,Merancang dan membangun aplikasi e-commerce li...,Merancang dan membangun aplikasi e-commerce li...,Awal entri belum dikenali.
3,proyek,L0020,BLoC untuk state management yang terstruktur.,BLoC untuk state management yang terstruktur.,Awal entri belum dikenali.
4,proyek,L0021,Mengintegrasikan payment gateway Midtrans untu...,Mengintegrasikan payment gateway Midtrans untu...,Awal entri belum dikenali.
5,proyek,L0022,Mengimplementasikan autentikasi aman berbasis ...,Mengimplementasikan autentikasi aman berbasis ...,Awal entri belum dikenali.
6,proyek,L0023,Sign-In.,Sign-In.,Awal entri belum dikenali.
7,proyek,L0024,Aplikasi Manajemen Keuangan Pribadi 'AturDuit'...,Aplikasi Manajemen Keuangan Pribadi 'AturDuit'...,Awal entri belum dikenali.
8,proyek,L0025,Oktober 2025 – Januari 2026,Oktober 2025 – Januari 2026,Awal entri belum dikenali.
9,proyek,L0026,Membangun aplikasi pencatat keuangan dengan Fl...,Membangun aplikasi pencatat keuangan dengan Fl...,Awal entri belum dikenali.


cell 8

In [11]:
def jalankan_pengujian():
    daftar_hasil = []
    def uji(nama, fungsi):
        try:
            fungsi()
            daftar_hasil.append({"pengujian": nama, "hasil": "LULUS", "detail": ""})
        except Exception as error:
            daftar_hasil.append({"pengujian": nama, "hasil": "GAGAL", "detail": f"{type(error).__name__}: {error}"})

    def test_batas_skill():
        _, _, h = proses_teks_uji("SKILLS\nJavaScript, GitHub")
        hasil_skill = skill_dari(h)
        assert hasil_skill == {"JavaScript", "GitHub"}

    def test_simbol():
        _, _, h = proses_teks_uji("Skills: C++, C#, NodeJS")
        hasil_skill = skill_dari(h)
        assert hasil_skill == {"C++", "C#", "Node.js"}

    def test_header_entri_fleksibel():
        teks = "PENDIDIKAN\nUniversitas Indonesia\n2018 - 2022\nPENGALAMAN\nPT Teknologi Jaya\nJanuari 2023 - Sekarang"
        sumber = buat_baris(teks, "uji", "txt")
        _, hasil_uji = analisis_baris(sumber)
        pendidikan = [e for e in hasil_uji["entri"] if e["bagian"] == "pendidikan"]
        pengalaman = [e for e in hasil_uji["entri"] if e["bagian"] == "pengalaman"]
        assert len(pendidikan) == 1, "Institusi dan tanggal terpecah"
        assert len(pengalaman) == 1, "Perusahaan dan tanggal terpecah"
        def nilai(entri, atribut): return [f["nilai_asli"] for f in entri["fields"] if f["atribut"] == atribut]
        assert nilai(pendidikan[0], "institusi") == ["Universitas Indonesia"]
        assert nilai(pendidikan[0], "periode") == ["2018 - 2022"]
        assert nilai(pengalaman[0], "perusahaan") == ["PT Teknologi Jaya"]
        assert nilai(pengalaman[0], "periode") == ["Januari 2023 - Sekarang"]
        assert len(pendidikan[0]["baris"]) == 2
        assert not hasil_uji["belum_terstruktur"], "Masih ada baris belum terstruktur"

    tests = [
        ("Batas nama Java/JavaScript dan Git/GitHub", test_batas_skill),
        ("Simbol C++, C# dan alias NodeJS", test_simbol),
        ("Deteksi header entri fleksibel (Tanggal/Institusi)", test_header_entri_fleksibel)
    ]
    for nama, fungsi in tests: uji(nama, fungsi)
    tabel = pd.DataFrame(daftar_hasil)
    display(tabel)
    print(f"Total: {len(daftar_hasil)}")
    return tabel

hasil_pengujian = jalankan_pengujian()

,pengujian,hasil,detail
0,Batas nama Java/JavaScript dan Git/GitHub,LULUS,
1,"Simbol C++, C# dan alias NodeJS",LULUS,
2,Deteksi header entri fleksibel (Tanggal/Instit...,LULUS,


Total: 3


In [12]:
# 4. Sel pengujian untuk bukti palsu
sumber_palsu = [{
    "id": "L0001",
    "teks": "PT Contoh",
}]

entri_palsu = [{
    "id_entri": "pengalaman_1",
    "bagian": "pengalaman",
    "baris": [{
        "baris_id": "L0001",
        "teks": "PT Contoh",
        "bukti": "PT Contoh",
    }],
    "fields": [{
        "atribut": "perusahaan",
        "nilai_asli": "PT Tidak Ada",
        "baris_id": "L0001",
        "bukti": "PT Contoh",
        "metode": "uji",
    }],
}]

cek_bukti = validasi_entri_profil(
    entri_palsu,
    sumber_palsu,
)

assert not cek_bukti[0]["fields"]
assert len(cek_bukti[0]["fields_ditahan"]) == 1
assert cek_bukti[0]["status"] == "perlu_perbaikan_bukti"

print("LULUS: field tanpa bukti yang sesuai berhasil ditahan.")

LULUS: field tanpa bukti yang sesuai berhasil ditahan.


In [13]:
# 4. Tambahkan satu sel baru untuk menguji bukti palsu

sumber_palsu = [{
    "id": "L0001",
    "teks": "PT Contoh",
}]

entri_palsu = [{
    "id_entri": "pengalaman_1",
    "bagian": "pengalaman",
    "baris": [{
        "baris_id": "L0001",
        "teks": "PT Contoh",
        "bukti": "PT Contoh",
    }],
    "fields": [{
        "atribut": "perusahaan",
        "nilai_asli": "PT Tidak Ada",
        "baris_id": "L0001",
        "bukti": "PT Contoh",
        "metode": "uji",
    }],
}]

cek_bukti = validasi_entri_profil(
    entri_palsu,
    sumber_palsu,
)

assert not cek_bukti[0]["fields"]
assert len(cek_bukti[0]["fields_ditahan"]) == 1
assert cek_bukti[0]["status"] == "perlu_perbaikan_bukti"

print("LULUS: field tanpa bukti yang sesuai berhasil ditahan.")

LULUS: field tanpa bukti yang sesuai berhasil ditahan.


cell 9

In [14]:
def hitung_metrik(prediksi, acuan):
    prediksi = set(prediksi)
    acuan = set(acuan)

    tp = len(prediksi & acuan)
    fp = len(prediksi - acuan)
    fn = len(acuan - prediksi)

    precision = tp / (tp + fp) if tp + fp else None
    recall = tp / (tp + fn) if tp + fn else None

    if precision is None or recall is None:
        f1 = None
    elif precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "prediksi_salah": sorted(prediksi - acuan),
        "terlewat": sorted(acuan - prediksi),
    }


TEKS_EVALUASI = """Nama: Budi Santoso
PROFIL PROFESIONAL
Lulusan informatika.
KEAHLIAN UTAMA
JavaScript, GitHub, PostgreSQL
PENGALAMAN PROYEK
Membangun dashboard menggunakan React.
PENGALAMAN KERJA / MAGANG
Perusahaan: PT Contoh
PENDIDIKAN
Institusi: Politeknik Contoh
SERTIFIKASI & PELATIHAN
Sertifikat Pengembangan Web
"""

# Jawaban acuan ditulis manual.
ACUAN_SKILL = {
    "JavaScript", "GitHub", "PostgreSQL", "React"
}

ACUAN_BAGIAN = {
    "L0001": "unclassified",
    "L0002": "ringkasan",
    "L0003": "ringkasan",
    "L0004": "keterampilan",
    "L0005": "keterampilan",
    "L0006": "proyek",
    "L0007": "proyek",
    "L0008": "pengalaman",
    "L0009": "pengalaman",
    "L0010": "pendidikan",
    "L0011": "pendidikan",
    "L0012": "sertifikasi",
    "L0013": "sertifikasi",
}

sumber_eval, bagian_eval, hasil_eval = proses_teks_uji(
    TEKS_EVALUASI
)

print("EVALUASI KEMUNCULAN SKILL")
display(pd.DataFrame([
    hitung_metrik(skill_dari(hasil_eval), ACUAN_SKILL)
]))

prediksi_bagian = {
    x["id"]: x["bagian"] for x in bagian_eval
}

perbandingan = [
    {
        "baris_id": baris_id,
        "acuan": kategori,
        "prediksi": prediksi_bagian.get(baris_id),
        "benar": prediksi_bagian.get(baris_id) == kategori,
    }
    for baris_id, kategori in ACUAN_BAGIAN.items()
]

print("\nEVALUASI PEMISAHAN BAGIAN")
display(pd.DataFrame(perbandingan))

akurasi_bagian = (
    sum(x["benar"] for x in perbandingan) / len(perbandingan)
    if perbandingan else None
)

print("Akurasi bagian pada contoh sintetis:", akurasi_bagian)
print(
    "Evaluasi ini belum mengukur kualitas OCR, "
    "ekstraksi format bebas, atau pemahaman negasi."
)

EVALUASI KEMUNCULAN SKILL


,TP,FP,FN,precision,recall,f1,prediksi_salah,terlewat
0,4,0,0,1.0,1.0,1.0,[],[]



EVALUASI PEMISAHAN BAGIAN


,baris_id,acuan,prediksi,benar
0,L0001,unclassified,unclassified,True
1,L0002,ringkasan,ringkasan,True
2,L0003,ringkasan,ringkasan,True
3,L0004,keterampilan,keterampilan,True
4,L0005,keterampilan,keterampilan,True
5,L0006,proyek,proyek,True
6,L0007,proyek,proyek,True
7,L0008,pengalaman,pengalaman,True
8,L0009,pengalaman,pengalaman,True
9,L0010,pendidikan,pendidikan,True


Akurasi bagian pada contoh sintetis: 1.0
Evaluasi ini belum mengukur kualitas OCR, ekstraksi format bebas, atau pemahaman negasi.


In [15]:
if not isinstance(hasil, dict):
    raise RuntimeError("Analisis belum menghasilkan profil.")

kunci_wajib = {"entri", "belum_terstruktur"}
kunci_hilang = kunci_wajib - hasil.keys()

if kunci_hilang:
    raise RuntimeError("Hasil belum memuat ekstraksi entri: " + ", ".join(sorted(kunci_hilang)))

print("Jumlah entri:", len(hasil["entri"]))
print("Jumlah baris belum terstruktur:", len(hasil["belum_terstruktur"]))

print("\nH. ENTRI PROFIL TERSTRUKTUR")
for entri in hasil.get("entri", []):
    print(f"\n{entri['id_entri']} — {entri['status']}")
    if entri["fields"]:
        display(pd.DataFrame(entri["fields"]))
    print("Baris terhubung:")
    display(pd.DataFrame(entri["baris"]))

print("\nI. BELUM TERSTRUKTUR")
if hasil.get("belum_terstruktur"):
    display(pd.DataFrame(hasil["belum_terstruktur"]))
else:
    print("Tidak ada.")

Jumlah entri: 2
Jumlah baris belum terstruktur: 15

H. ENTRI PROFIL TERSTRUKTUR

pengalaman_1 — kandidat_perlu_review


,atribut,nilai_asli,baris_id,bukti,metode,nilai_normal,status
0,perusahaan,PT Nusantara Digital Kreatif,L0031,PT Nusantara Digital Kreatif — Junior Mobile D...,pola_header,PT Nusantara Digital Kreatif,bukti_cocok_perlu_review
1,jabatan,Junior Mobile Developer (Internship),L0031,PT Nusantara Digital Kreatif — Junior Mobile D...,pola_header,Junior Mobile Developer (Internship),bukti_cocok_perlu_review
2,periode,Juli 2025 – September 2025,L0032,Juli 2025 – September 2025,pola_periode,Juli 2025 – September 2025,bukti_cocok_perlu_review


Baris terhubung:


,baris_id,teks,bukti
0,L0031,PT Nusantara Digital Kreatif — Junior Mobile D...,PT Nusantara Digital Kreatif — Junior Mobile D...
1,L0032,Juli 2025 – September 2025,Juli 2025 – September 2025
2,L0033,Berkolaborasi dengan tim engineering dalam pem...,Berkolaborasi dengan tim engineering dalam pem...
3,L0034,perusahaan.,perusahaan.
4,L0035,Mengembangkan fitur notifikasi push dan pembar...,Mengembangkan fitur notifikasi push dan pembar...
5,L0036,Flutter.,Flutter.
6,L0037,"Berpartisipasi aktif dalam ritual Agile/Scrum,...","Berpartisipasi aktif dalam ritual Agile/Scrum,..."



pendidikan_1 — kandidat_perlu_review


,atribut,nilai_asli,baris_id,bukti,metode,nilai_normal,status
0,institusi,Universitas Bina Cendekia,L0039,Universitas Bina Cendekia — S1 Teknik Informatika,pola_header,Universitas Bina Cendekia,bukti_cocok_perlu_review
1,jenjang,S1,L0039,Universitas Bina Cendekia — S1 Teknik Informatika,pola_header,S1,bukti_cocok_perlu_review
2,program_studi,Teknik Informatika,L0039,Universitas Bina Cendekia — S1 Teknik Informatika,pola_header,Teknik Informatika,bukti_cocok_perlu_review
3,periode,2022 – 2026,L0040,2022 – 2026,pola_periode,2022 – 2026,bukti_cocok_perlu_review
4,ipk,3.80/4.00 — Judul Skripsi: Implementasi Arsite...,L0041,IPK: 3.80/4.00 — Judul Skripsi: Implementasi A...,label_eksplisit,3.80/4.00 — Judul Skripsi: Implementasi Arsite...,bukti_cocok_perlu_review


Baris terhubung:


,baris_id,teks,bukti
0,L0039,Universitas Bina Cendekia — S1 Teknik Informatika,Universitas Bina Cendekia — S1 Teknik Informatika
1,L0040,2022 – 2026,2022 – 2026
2,L0041,IPK: 3.80/4.00 — Judul Skripsi: Implementasi A...,IPK: 3.80/4.00 — Judul Skripsi: Implementasi A...
3,L0042,Transportasi Real-Time,Transportasi Real-Time



I. BELUM TERSTRUKTUR


,bagian,baris_id,teks,bukti,alasan
0,proyek,L0017,"Aplikasi E-Commerce ""BelanjaYuk"" — Mobile Deve...","Aplikasi E-Commerce ""BelanjaYuk"" — Mobile Deve...",Awal entri belum dikenali.
1,proyek,L0018,April 2026 – Agustus 2026,April 2026 – Agustus 2026,Awal entri belum dikenali.
2,proyek,L0019,Merancang dan membangun aplikasi e-commerce li...,Merancang dan membangun aplikasi e-commerce li...,Awal entri belum dikenali.
3,proyek,L0020,BLoC untuk state management yang terstruktur.,BLoC untuk state management yang terstruktur.,Awal entri belum dikenali.
4,proyek,L0021,Mengintegrasikan payment gateway Midtrans untu...,Mengintegrasikan payment gateway Midtrans untu...,Awal entri belum dikenali.
5,proyek,L0022,Mengimplementasikan autentikasi aman berbasis ...,Mengimplementasikan autentikasi aman berbasis ...,Awal entri belum dikenali.
6,proyek,L0023,Sign-In.,Sign-In.,Awal entri belum dikenali.
7,proyek,L0024,Aplikasi Manajemen Keuangan Pribadi 'AturDuit'...,Aplikasi Manajemen Keuangan Pribadi 'AturDuit'...,Awal entri belum dikenali.
8,proyek,L0025,Oktober 2025 – Januari 2026,Oktober 2025 – Januari 2026,Awal entri belum dikenali.
9,proyek,L0026,Membangun aplikasi pencatat keuangan dengan Fl...,Membangun aplikasi pencatat keuangan dengan Fl...,Awal entri belum dikenali.


In [16]:
# MUAT MODEL — JALANKAN SEKALI SETIAP SESI COLAB

import io
import joblib
import sklearn
from google.colab import files

unggahan_model = files.upload()

nama_file_model = [
    nama for nama in unggahan_model
    if nama.lower().endswith(".joblib")
]

if len(nama_file_model) != 1:
    raise ValueError("Upload tepat satu file model .joblib.")

paket_classifier = joblib.load(
    io.BytesIO(unggahan_model[nama_file_model[0]])
)

kolom_wajib = {
    "pipeline", "sklearn_version", "kategori", "normalisasi"
}

if (
    not isinstance(paket_classifier, dict)
    or not kolom_wajib.issubset(paket_classifier)
):
    raise ValueError("Format paket model tidak sesuai dengan Sel 8.")

versi_training = paket_classifier["sklearn_version"]
versi_saat_ini = sklearn.__version__


def _versi_utama_minor(v):
    bagian = v.split(".")
    return tuple(int(p) for p in bagian[:2])


if _versi_utama_minor(versi_training)[0] != _versi_utama_minor(versi_saat_ini)[0]:
    # Beda versi MAJOR (mis. scikit-learn 1.x vs 2.x) jauh lebih rawan
    # tidak kompatibel, jadi ini tetap diblokir.
    raise RuntimeError(
        f"Versi scikit-learn berbeda pada angka MAJOR. "
        f"Training: {versi_training}; notebook ini: {versi_saat_ini}. "
        "Latih ulang model dengan versi saat ini, atau samakan versi lingkungan."
    )
elif versi_training != versi_saat_ini:
    # Beda versi MINOR/PATCH biasanya masih aman untuk model TF-IDF +
    # Logistic Regression sederhana seperti ini — beri peringatan saja,
    # jangan gagalkan seluruh notebook.
    print(
        f"⚠️ Peringatan: model dilatih dengan scikit-learn {versi_training}, "
        f"lingkungan saat ini {versi_saat_ini}. Melanjutkan pemuatan model; "
        "periksa kembali hasil prediksi bila terlihat janggal."
    )

if paket_classifier["normalisasi"] != "lowercase_whitespace_v1":
    raise ValueError("Metode normalisasi model tidak dikenali.")

classifier_cv = paket_classifier["pipeline"]

print("Model berhasil dimuat.")
print("Jumlah kategori:", len(paket_classifier["kategori"]))
print("Siap digunakan tanpa training ulang.")

Saving job_role_classifier.joblib to job_role_classifier (1).joblib
Model berhasil dimuat.
Jumlah kategori: 30
Siap digunakan tanpa training ulang.


In [17]:
# PREDIKSI KATEGORI DARI TEKS CV

import re
import numpy as np
import pandas as pd
from IPython.display import display


def prediksi_posisi_cv(teks_cv, top_k=3):
    if "classifier_cv" not in globals():
        raise RuntimeError("Jalankan sel pemuatan model terlebih dahulu.")

    if not isinstance(teks_cv, str) or not teks_cv.strip():
        raise ValueError("Teks CV kosong atau bukan string.")

    # Sama dengan normalisasi pada notebook training.
    teks_bersih = re.sub(r"\s+", " ", teks_cv.lower()).strip()

    if len(teks_bersih.split()) < 30:
        raise ValueError(
            "Teks terlalu pendek untuk percobaan ini. "
            "Periksa apakah isi CV sudah terbaca lengkap."
        )

    fitur = classifier_cv.named_steps["tfidf"].transform(
        [teks_bersih]
    )

    if fitur.nnz == 0:
        raise ValueError(
            "Tidak ada istilah CV yang dikenali model."
        )

    probabilitas = classifier_cv.predict_proba([teks_bersih])[0]
    kategori = classifier_cv.named_steps["classifier"].classes_

    top_k = max(1, min(int(top_k), len(kategori)))
    indeks = np.argsort(-probabilitas)[:top_k]

    return pd.DataFrame({
        "peringkat": np.arange(1, top_k + 1),
        "kategori_prediksi": kategori[indeks],
        "skor_model": probabilitas[indeks],
    })


print("Fungsi prediksi siap.")

Fungsi prediksi siap.


In [18]:
# PREDIKSI OTOMATIS DARI CV YANG DIUPLOAD

hasil_prediksi_posisi = None

if not isinstance(globals().get("hasil"), dict):
    raise RuntimeError("Jalankan sel upload dan analisis CV dahulu.")

if not globals().get("sumber_baris"):
    raise RuntimeError("Teks hasil pembacaan CV belum tersedia.")

teks_cv_untuk_prediksi = "\n".join(
    baris["teks"]
    for baris in sumber_baris
    if isinstance(baris.get("teks"), str)
    and baris["teks"].strip()
)

print("CV:", globals().get("nama_cv", "CV aktif"))
print("Jumlah kata:", len(teks_cv_untuk_prediksi.split()))

hasil_prediksi_posisi = prediksi_posisi_cv(
    teks_cv_untuk_prediksi,
    top_k=3,
)

print("\nPETUNJUK KATEGORI DARI CLASSIFIER")
display(hasil_prediksi_posisi)

print(
    "Kategori classifier merupakan petunjuk tambahan. "
    "Rekomendasi berbasis kecocokan skill ditampilkan terpisah."
)

CV: CV_Kesha_Amelia_Mobile_Developer.pdf
Jumlah kata: 315

PETUNJUK KATEGORI DARI CLASSIFIER


,peringkat,kategori_prediksi,skor_model
0,1,Mobile Developer,0.496590
1,2,React Developer,0.108104
2,3,Python Developer,0.053313


Kategori classifier merupakan petunjuk tambahan. Rekomendasi berbasis kecocokan skill ditampilkan terpisah.


In [21]:
# MEMUAT REFERENSI POSISI DAN KEBUTUHAN SKILL

import io
import pandas as pd
from google.colab import files

referensi_job = None
unggahan_job = files.upload()

csv_files = [
    nama for nama in unggahan_job
    if nama.lower().endswith(".csv")
]

if len(csv_files) != 1:
    raise ValueError("Upload tepat satu CSV: job_roles.csv.")

job_df = pd.read_csv(
    io.BytesIO(unggahan_job[csv_files[0]])
)

kolom_wajib = {"Job Title", "Category", "Required Skills"}

if not kolom_wajib.issubset(job_df.columns):
    raise ValueError(
        "CSV harus memiliki Job Title, Category, dan Required Skills."
    )

# Cakupan awal: teknologi dan bidang data.
referensi_job = job_df.loc[
    job_df["Category"].isin(["Technology", "Data & Analytics"])
].copy()

if referensi_job.empty:
    raise ValueError("Tidak ditemukan kategori teknologi/data.")

if referensi_job[list(kolom_wajib)].isna().any().any():
    raise ValueError("Referensi posisi memiliki kolom wajib yang kosong.")

if referensi_job["Job Title"].duplicated().any():
    raise ValueError("Ada nama posisi duplikat; tinjau referensinya.")

referensi_job = referensi_job.reset_index(drop=True)

print("Referensi siap:", len(referensi_job), "posisi.")
display(referensi_job[["Job Title", "Required Skills"]].head())

Saving job_roles.csv to job_roles (2).csv
Referensi siap: 101 posisi.


,Job Title,Required Skills
0,Software Engineer,Python|Java|C++|Git|Software Design|Problem So...
1,Full Stack Developer,JavaScript|React|Node.js|HTML/CSS|Database|Git
2,Frontend Developer,JavaScript|React|Vue.js|CSS|HTML|UI/UX Design
3,Backend Developer,Python|Java|Node.js|Database Design|API Develo...
4,DevOps Engineer,Docker|Kubernetes|AWS|Linux|CI/CD|Scripting


In [22]:
# REKOMENDASI POSISI BERDASARKAN BUKTI CV — VERSI 2

import re
import unicodedata
import pandas as pd
from IPython.display import display


def normal_rekom(teks):
    teks = unicodedata.normalize("NFKC", str(teks))
    return re.sub(r"\s+", " ", teks.casefold()).strip()


# Alias berarti nama lain, bukan otomatis kemampuan yang setara.
ALIAS_REKOM = {
    "JavaScript": ["javascript", "java script"],
    "TypeScript": ["typescript"],
    "Python": ["python"],
    "Java": ["java"],
    "React": ["react", "react.js", "reactjs"],
    "Vue.js": ["vue.js", "vuejs", "vue"],
    "Angular": ["angular"],
    "Node.js": ["node.js", "nodejs", "node js"],
    "Express.js": ["express.js", "expressjs"],
    "Django": ["django"],
    "Flask": ["flask"],
    "HTML": ["html", "html5"],
    "CSS": ["css", "css3"],
    "Tailwind CSS": ["tailwind css", "tailwindcss", "tailwind"],
    "SQL": ["sql"],
    "PostgreSQL": ["postgresql", "postgres"],
    "MySQL": ["mysql"],
    "MongoDB": ["mongodb"],
    "Git": ["git"],
    "REST API": ["rest api", "restful api", "restful apis"],
    "API Development": ["api development", "pengembangan api"],
    "API Integration": ["api integration", "integrasi api"],
    "Database": ["database", "basis data"],
    "Database Design": ["database design", "perancangan basis data"],
    "Docker": ["docker"],
    "Kubernetes": ["kubernetes", "k8s"],
    "AWS": ["aws", "amazon web services"],
    "Linux": ["linux"],
    "CI/CD": ["ci/cd", "ci cd"],
    "Redux": ["redux"],
    "Vuex": ["vuex"],
    "RxJS": ["rxjs"],
    "Spring Boot": ["spring boot", "springboot"],
    "Machine Learning": ["machine learning", "pembelajaran mesin"],
    "Deep Learning": ["deep learning"],
    "TensorFlow": ["tensorflow"],
    "PyTorch": ["pytorch"],
    "Spark": ["spark", "apache spark"],
    "Hadoop": ["hadoop"],
    "ETL": ["etl"],
}


# Hubungan terbatas dan transparan.
# Tidak disamakan dengan alias atau bukti tingkat penguasaan.
RELASI_REKOM = {
    "Database": ["PostgreSQL", "MySQL", "MongoDB"],
}


# Setiap kelompok harus memiliki minimal satu skill berbukti.
# Ini aturan baseline untuk rekomendasi, bukan syarat hiring.
ATURAN_UTAMA = {
    "Frontend Developer": [
        ["JavaScript", "TypeScript"],
        ["React", "Vue.js", "Angular", "HTML"],
    ],
    "Backend Developer": [
        ["Python", "Java", "Node.js"],
        ["API Development", "REST API"],
    ],
    "Full Stack Developer": [
        ["React", "Vue.js", "Angular", "HTML"],
        ["Node.js", "Django", "Flask", "Spring Boot"],
        ["Database"],
    ],
    "React Developer": [["React"]],
    "Vue Developer": [["Vue.js"]],
    "Angular Developer": [["Angular"]],
    "Python Developer": [["Python"]],
    "Java Backend Developer": [["Java"]],
    "DevOps Engineer": [
        ["Docker", "Kubernetes", "CI/CD"],
        ["Linux", "AWS"],
    ],
    "Software Engineer": [["Python", "Java", "C++"]],
    "Data Scientist": [
        ["Python"],
        ["Machine Learning", "Statistics", "Data Analysis"],
    ],
    "Data Engineer": [
        ["SQL"],
        ["ETL", "Spark", "Hadoop", "Data Warehousing"],
    ],
}


# Baseline konservatif: baris dengan konteks ini ditahan untuk review.
# Tidak mengklaim sudah memahami semua bentuk negasi.
POLA_REVIEW = re.compile(
    r"\b(?:belum|tidak|tanpa|ingin|berencana|"
    r"not|never|without|lack|lacking)\b"
    r"|no experience|want to learn|plan to learn",
    re.IGNORECASE,
)


def rekomendasi_v2(baris_cv, referensi, kamus_lama):
    # Gabungkan kamus lama dengan alias yang diperjelas.
    kamus = {
        nama: list(dict.fromkeys([nama, *alias]))
        for nama, alias in kamus_lama.items()
    }

    for nama, alias in ALIAS_REKOM.items():
        kamus[nama] = list(dict.fromkeys([nama, *alias]))

    # Tambahkan pencarian literal kebutuhan referensi.
    # Pencarian literal belum menjamin cakupan bahasa/alias lengkap.
    for daftar in referensi["Required Skills"]:
        for nama in str(daftar).split("|"):
            nama = nama.strip()
            if nama and "/" not in nama:
                kamus.setdefault(nama, [nama])

    # HTML/CSS didefinisikan sebagai gabungan keduanya.
    kebutuhan_gabungan = {"HTML/CSS": ["HTML", "CSS"]}

    pola = {
        nama: re.compile(
            r"(?<![\w+#])(?:"
            + "|".join(
                re.escape(normal_rekom(a))
                for a in sorted(aliases, key=len, reverse=True)
            )
            + r")(?![\w+#])"
        )
        for nama, aliases in kamus.items()
    }

    bukti = {}
    ditahan = []

    def tambah(skill, item):
        bukti.setdefault(skill, []).append(item)

    for baris in baris_cv:
        if baris.get("is_heading"):
            continue

        bagian = baris.get("bagian", "unclassified")

        # Identitas tidak menjadi bukti skill.
        if bagian == "identitas":
            continue

        teks_asli = str(baris.get("isi", ""))
        teks = normal_rekom(teks_asli)

        if not teks:
            continue

        item = {
            "baris_id": baris.get("id"),
            "bagian": bagian,
            "bukti": teks_asli,
            "metode": "penyebutan langsung",
        }

        perlu_review = bool(POLA_REVIEW.search(teks))

        for skill, regex in pola.items():
            teks_pencarian = teks

            if skill == "CSS":
                # Hilangkan frasa Tailwind CSS sebelum mencari CSS.
                teks_pencarian = re.sub(
                    r"\btailwind\s+css\b", "tailwind", teks_pencarian
                )

            if regex.search(teks_pencarian):
                if perlu_review:
                    ditahan.append({
                        "skill": skill,
                        **item,
                        "alasan": "Konteks negasi/niat perlu ditinjau.",
                    })
                else:
                    tambah(skill, item.copy())

        # Bukti tindakan API: tetap berupa aturan pola, bukan pemahaman LLM.
        if not perlu_review:
            ada_api = bool(re.search(r"\bapi\b", teks))

            tindakan_api = {
                "API Development": (
                    r"\b(membangun|mengembangkan|membuat|"
                    r"developed|developing|built|implemented)\b"
                ),
                "API Integration": (
                    r"\b(mengintegrasikan|integrasi|"
                    r"integrated|integrating)\b"
                ),
            }

            for skill, tindakan in tindakan_api.items():
                if ada_api and re.search(tindakan, teks):
                    tambah(skill, {
                        **item,
                        "metode": "pola tindakan API; perlu review",
                    })

    def cari_bukti(skill):
        if skill in kebutuhan_gabungan:
            komponen = kebutuhan_gabungan[skill]
            hasil_komponen = [cari_bukti(s) for s in komponen]

            if not all(hasil_komponen):
                return []

            return [
                b
                for kumpulan in hasil_komponen
                for b in kumpulan
            ]

        hasil_skill = list(bukti.get(skill, []))

        for teknologi in RELASI_REKOM.get(skill, []):
            for b in bukti.get(teknologi, []):
                hasil_skill.append({
                    **b,
                    "metode": f"bukti terkait: {teknologi} → {skill}",
                })

        return hasil_skill

    hasil_posisi = []
    detail = {}
    ditunda = []

    for _, job in referensi.iterrows():
        posisi = job["Job Title"]

        if posisi not in ATURAN_UTAMA:
            continue

        kelompok_hilang = [
            " / ".join(kelompok)
            for kelompok in ATURAN_UTAMA[posisi]
            if not any(cari_bukti(s) for s in kelompok)
        ]

        if kelompok_hilang:
            ditunda.append({
                "posisi": posisi,
                "alasan": "Bukti skill utama belum ditemukan",
                "kelompok_skill": "; ".join(kelompok_hilang),
            })
            continue

        kebutuhan = list(dict.fromkeys(
            s.strip()
            for s in str(job["Required Skills"]).split("|")
            if s.strip()
        ))

        if not kebutuhan:
            continue

        cocok = []
        belum = []
        bukti_posisi = []
        jumlah_proyek_pengalaman = 0

        for skill in kebutuhan:
            temuan = cari_bukti(skill)

            if not temuan:
                belum.append(skill)
                continue

            cocok.append(skill)

            if any(
                b["bagian"] in {"proyek", "pengalaman"}
                for b in temuan
            ):
                jumlah_proyek_pengalaman += 1

            bukti_posisi.extend(
                {"kebutuhan": skill, **b} for b in temuan
            )

        if not cocok:
            continue

        hasil_posisi.append({
            "posisi": posisi,
            "cakupan_bukti_persen": 100 * len(cocok) / len(kebutuhan),
            "kebutuhan_cocok": len(cocok),
            "total_kebutuhan": len(kebutuhan),
            "didukung_proyek_pengalaman": jumlah_proyek_pengalaman,
        })

        detail[posisi] = {
            "cocok": cocok,
            "belum_ditemukan": belum,
            "bukti": bukti_posisi,
        }

    tabel = pd.DataFrame(hasil_posisi)

    if not tabel.empty:
        # Proyek/pengalaman hanya menjadi pembeda saat cakupan sama.
        tabel = tabel.sort_values(
            ["cakupan_bukti_persen", "didukung_proyek_pengalaman"],
            ascending=[False, False],
            kind="stable",
        ).reset_index(drop=True)

        pasangan = list(zip(
            tabel["cakupan_bukti_persen"],
            tabel["didukung_proyek_pengalaman"],
        ))

        urutan_unik = list(dict.fromkeys(pasangan))
        peringkat = {nilai: i + 1 for i, nilai in enumerate(urutan_unik)}

        tabel.insert(0, "peringkat", [
            peringkat[nilai] for nilai in pasangan
        ])

        tabel["cakupan_bukti_persen"] = (
            tabel["cakupan_bukti_persen"].round(2)
        )

    belum_diatur = sorted(
        set(referensi["Job Title"]) - set(ATURAN_UTAMA)
    )

    return tabel, detail, pd.DataFrame(ditunda), ditahan, belum_diatur


# Pastikan hasil berasal dari upload CV aktif.
if not globals().get("baris_terpisah"):
    raise RuntimeError("Jalankan upload dan analisis CV terlebih dahulu.")

if not isinstance(globals().get("referensi_job"), pd.DataFrame):
    raise RuntimeError("Upload job_roles.csv terlebih dahulu.")

if "KAMUS_SKILL" not in globals():
    raise RuntimeError("Jalankan sel kamus skill terlebih dahulu.")

(
    rekomendasi_terbaru,
    rincian_terbaru,
    posisi_ditunda,
    bukti_perlu_review,
    posisi_belum_diatur,
) = rekomendasi_v2(
    baris_cv=baris_terpisah,
    referensi=referensi_job,
    kamus_lama=KAMUS_SKILL,
)

print("CV:", globals().get("nama_cv", "CV aktif"))
print("Posisi dengan aturan:", len(ATURAN_UTAMA))

if rekomendasi_terbaru.empty:
    print(
        "Belum ada rekomendasi yang memenuhi aturan bukti utama. "
        "Periksa pembacaan CV, alias, dan konteks skill."
    )
else:
    # Bisa lebih dari 3 posisi jika ada peringkat yang sama.
    tampil = rekomendasi_terbaru.loc[
        rekomendasi_terbaru["peringkat"] <= 3
    ]

    print("\nPOSISI POTENSIAL — TIGA PERINGKAT TERATAS")
    display(tampil)

    for posisi in tampil["posisi"]:
        d = rincian_terbaru[posisi]

        print(f"\n{posisi}")
        print("Kebutuhan dengan bukti:", ", ".join(d["cocok"]))

        display(pd.DataFrame(d["bukti"]).drop_duplicates())

        print(
            "Belum ditemukan di CV:",
            ", ".join(d["belum_ditemukan"]) or "Tidak ada.",
        )

        if d["belum_ditemukan"]:
            print(
                "Saran: jika sudah dikuasai, tambahkan bukti relevan "
                "ke CV. Jika belum, pertimbangkan pembelajaran atau "
                "proyek sesuai posisi yang ingin dituju."
            )

print("\nPOSISI DENGAN BUKTI UTAMA BELUM CUKUP")
display(posisi_ditunda)

if bukti_perlu_review:
    print("\nPENYEBUTAN YANG DITAHAN UNTUK REVIEW")
    display(pd.DataFrame(bukti_perlu_review).drop_duplicates())

print(
    "\nPosisi referensi yang belum memiliki aturan:",
    len(posisi_belum_diatur),
)
print(", ".join(posisi_belum_diatur))

print(
    "\nSkor adalah cakupan kebutuhan skill berbasis aturan. "
    "Bukan probabilitas diterima atau ukuran tingkat penguasaan. "
    "Pengalaman proyek menjadi bukti pendukung, bukan verifikasi kemampuan."
)

CV: CV_Kesha_Amelia_Mobile_Developer.pdf
Posisi dengan aturan: 12

POSISI POTENSIAL — TIGA PERINGKAT TERATAS


,peringkat,posisi,cakupan_bukti_persen,kebutuhan_cocok,total_kebutuhan,didukung_proyek_pengalaman
0,1,React Developer,50.00,3,6,0
1,2,Frontend Developer,33.33,2,6,0



React Developer
Kebutuhan dengan bukti: JavaScript, React, API Integration


,kebutuhan,baris_id,bagian,bukti,metode
0,JavaScript,L0011,keterampilan,"Bahasa Pemrograman: Dart, Kotlin, Swift, JavaS...",penyebutan langsung
1,React,L0012,keterampilan,"Framework & Library: Flutter, React Native, Fi...",penyebutan langsung
2,API Integration,L0008,ringkasan,mengintegrasikan RESTful API dan layanan berba...,pola tindakan API; perlu review


Belum ditemukan di CV: CSS, HTML, Redux
Saran: jika sudah dikuasai, tambahkan bukti relevan ke CV. Jika belum, pertimbangkan pembelajaran atau proyek sesuai posisi yang ingin dituju.

Frontend Developer
Kebutuhan dengan bukti: JavaScript, React


,kebutuhan,baris_id,bagian,bukti,metode
0,JavaScript,L0011,keterampilan,"Bahasa Pemrograman: Dart, Kotlin, Swift, JavaS...",penyebutan langsung
1,React,L0012,keterampilan,"Framework & Library: Flutter, React Native, Fi...",penyebutan langsung


Belum ditemukan di CV: Vue.js, CSS, HTML, UI/UX Design
Saran: jika sudah dikuasai, tambahkan bukti relevan ke CV. Jika belum, pertimbangkan pembelajaran atau proyek sesuai posisi yang ingin dituju.

POSISI DENGAN BUKTI UTAMA BELUM CUKUP


,posisi,alasan,kelompok_skill
0,Software Engineer,Bukti skill utama belum ditemukan,Python / Java / C++
1,Full Stack Developer,Bukti skill utama belum ditemukan,Node.js / Django / Flask / Spring Boot
2,Backend Developer,Bukti skill utama belum ditemukan,Python / Java / Node.js
3,DevOps Engineer,Bukti skill utama belum ditemukan,Linux / AWS
4,Data Scientist,Bukti skill utama belum ditemukan,Python; Machine Learning / Statistics / Data A...
5,Data Engineer,Bukti skill utama belum ditemukan,ETL / Spark / Hadoop / Data Warehousing
6,Vue Developer,Bukti skill utama belum ditemukan,Vue.js
7,Angular Developer,Bukti skill utama belum ditemukan,Angular
8,Python Developer,Bukti skill utama belum ditemukan,Python
9,Java Backend Developer,Bukti skill utama belum ditemukan,Java



Posisi referensi yang belum memiliki aturan: 89
AI/ML Specialist, AR/VR Developer, Academic Data Analyst, Analytics Engineer, BI Analyst, BI Developer, Blockchain Developer, Business Analyst, C# Developer, C++ Developer, Cassandra Developer, Clojure Developer, Cloud Architect, Compliance Officer, Cybersecurity Analyst, Cybersecurity Specialist, Data Analyst, Data Warehouse Architect, Database Administrator, DevSecOps Engineer, Drupal Developer, ETL Developer, Elasticsearch Developer, Elixir Developer, Enterprise Architect, Ethical Hacker, Game Developer, Go Developer, Hadoop Developer, Haskell Developer, IT Field Technician, IT Help Desk Technician, IT Project Manager, Informatica Developer, IoT Engineer, Joomla Developer, Jupyter Developer, Kafka Developer, Kotlin Developer, Looker Developer, MATLAB Developer, Machine Learning Engineer, Magento Developer, Mobile Developer Android, Mobile Developer iOS, MongoDB Developer, Network Engineer, Oracle Applications Developer, Oracle DBA, PH

In [23]:
# REKOMENDASI V3 — KELOMPOK KEBUTUHAN DAN KONTEKS BUKTI

import re
import pandas as pd
from IPython.display import display

if not callable(globals().get("rekomendasi_v2")):
    raise RuntimeError("Jalankan sel rekomendasi V2 terlebih dahulu.")

if not globals().get("baris_terpisah"):
    raise RuntimeError("Upload dan analisis CV terlebih dahulu.")

if not isinstance(globals().get("referensi_job"), pd.DataFrame):
    raise RuntimeError("Upload job_roles.csv terlebih dahulu.")


# Struktur kelompok:
# (nama kelompok, pilihan skill, jenis)
#
# Dalam satu kelompok, cukup ada SATU alternatif.
# Semua kelompok "utama" harus memiliki bukti untuk ditampilkan
# sebagai rekomendasi awal.
#
# Setiap kelompok dihitung satu kali, tanpa bobot buatan.
PROFIL_POSISI_V3 = {
    "Full Stack Developer": [
        ("Bahasa frontend", ["JavaScript", "TypeScript"], "utama"),
        ("Framework frontend", ["React", "Vue.js", "Angular"], "utama"),
        ("Teknologi backend", ["Node.js", "Django", "Flask", "Spring Boot"], "utama"),
        ("Database", ["Database"], "utama"),
        ("HTML", ["HTML"], "pendukung"),
        ("CSS", ["CSS"], "pendukung"),
        ("Version control", ["Git"], "pendukung"),
    ],
    "Frontend Developer": [
        ("Bahasa frontend", ["JavaScript", "TypeScript"], "utama"),
        ("Framework frontend", ["React", "Vue.js", "Angular"], "utama"),
        ("HTML", ["HTML"], "pendukung"),
        ("CSS", ["CSS"], "pendukung"),
        ("Integrasi API", ["API Integration"], "pendukung"),
        ("Version control", ["Git"], "pendukung"),
    ],
    "Backend Developer": [
        ("Bahasa/backend runtime", ["Python", "Java", "Node.js"], "utama"),
        ("Pengembangan API", ["API Development"], "utama"),
        ("Database", ["Database", "SQL"], "utama"),
        ("Desain database", ["Database Design"], "pendukung"),
        ("Version control", ["Git"], "pendukung"),
        ("Kontainer", ["Docker"], "pendukung"),
    ],
    "Python Developer": [
        ("Bahasa Python", ["Python"], "utama"),
        ("Framework web", ["Django", "Flask"], "pendukung"),
        ("SQL", ["SQL"], "pendukung"),
        ("API", ["REST API", "API Development"], "pendukung"),
        ("Version control", ["Git"], "pendukung"),
    ],
    "React Developer": [
        ("React", ["React"], "utama"),
        ("Bahasa frontend", ["JavaScript", "TypeScript"], "utama"),
        ("HTML", ["HTML"], "pendukung"),
        ("CSS", ["CSS"], "pendukung"),
        ("State management", ["Redux"], "pendukung"),
        ("Integrasi API", ["API Integration"], "pendukung"),
    ],
    "Vue Developer": [
        ("Vue", ["Vue.js"], "utama"),
        ("Bahasa frontend", ["JavaScript", "TypeScript"], "utama"),
        ("HTML", ["HTML"], "pendukung"),
        ("CSS", ["CSS"], "pendukung"),
        ("State management", ["Vuex"], "pendukung"),
        ("Integrasi API", ["API Integration"], "pendukung"),
    ],
    "Angular Developer": [
        ("Angular", ["Angular"], "utama"),
        ("TypeScript", ["TypeScript"], "utama"),
        ("HTML", ["HTML"], "pendukung"),
        ("CSS", ["CSS"], "pendukung"),
        ("Reactive programming", ["RxJS"], "pendukung"),
        ("Integrasi API", ["API Integration"], "pendukung"),
    ],
    "Java Backend Developer": [
        ("Java", ["Java"], "utama"),
        ("Pengembangan API", ["API Development"], "utama"),
        ("Framework", ["Spring Boot"], "pendukung"),
        ("Database", ["SQL", "Database"], "pendukung"),
        ("Version control", ["Git"], "pendukung"),
    ],
    "Software Engineer": [
        ("Bahasa pemrograman", ["Python", "Java", "C++", "JavaScript"], "utama"),
        ("Version control", ["Git"], "pendukung"),
        ("Desain perangkat lunak", ["Software Design"], "pendukung"),
        ("Pengujian", ["Test Automation", "Manual Testing"], "pendukung"),
        ("Problem solving", ["Problem Solving"], "pendukung"),
    ],
    "DevOps Engineer": [
        ("Linux", ["Linux"], "utama"),
        ("Otomasi delivery", ["CI/CD"], "utama"),
        ("Kontainer", ["Docker", "Kubernetes"], "utama"),
        ("Cloud", ["AWS", "GCP", "Azure"], "pendukung"),
        ("Scripting", ["Scripting", "Python", "Shell Scripting"], "pendukung"),
    ],
    "Data Scientist": [
        ("Bahasa analisis", ["Python"], "utama"),
        ("Machine learning", ["Machine Learning"], "utama"),
        ("Statistik", ["Statistics", "Statistical Analysis"], "utama"),
        ("SQL", ["SQL"], "pendukung"),
        ("Framework ML", ["TensorFlow", "PyTorch"], "pendukung"),
        ("Analisis data", ["Data Analysis"], "pendukung"),
    ],
    "Data Engineer": [
        ("SQL", ["SQL"], "utama"),
        ("Pipeline data", ["ETL", "Spark", "Hadoop"], "utama"),
        ("Python", ["Python"], "pendukung"),
        ("Data warehouse", ["Data Warehousing"], "pendukung"),
        ("Database", ["Database"], "pendukung"),
    ],
}


# Periksa bahwa profil kurasi menunjuk posisi yang benar-benar
# tersedia dalam dataset utama.
posisi_tidak_tersedia = (
    set(PROFIL_POSISI_V3)
    - set(referensi_job["Job Title"])
)

if posisi_tidak_tersedia:
    raise ValueError(
        f"Posisi kurasi tidak ada dalam referensi: "
        f"{sorted(posisi_tidak_tersedia)}"
    )


# Tambahan alias lokal: tidak mengubah KAMUS_SKILL parser.
kamus_v3 = {
    nama: list(aliases)
    for nama, aliases in KAMUS_SKILL.items()
}

kamus_v3.update({
    "GCP": ["gcp", "google cloud platform"],
    "Azure": ["azure", "microsoft azure"],
    "Software Design": ["software design", "desain perangkat lunak"],
    "Problem Solving": ["problem solving", "pemecahan masalah"],
    "Manual Testing": ["manual testing", "pengujian manual"],
    "Test Automation": ["test automation", "automated testing"],
})


# Gunakan mesin bukti V2, tetapi kumpulkan bukti semua posisi
# terlebih dahulu. Filter kelompok utama dilakukan oleh V3.
#
# Referensi sementara hanya dipakai dalam memori untuk pencarian
# bukti; job_roles.csv asli tidak diubah.
referensi_bukti = referensi_job.copy()

for posisi, kelompok in PROFIL_POSISI_V3.items():
    skill_profil = list(dict.fromkeys(
        skill
        for _, pilihan, _ in kelompok
        for skill in pilihan
    ))

    referensi_bukti.loc[
        referensi_bukti["Job Title"] == posisi,
        "Required Skills",
    ] = "|".join(skill_profil)


# Fungsi V2 membaca ATURAN_UTAMA global.
# Simpan lalu pulihkan agar aturan V2 tidak berubah permanen.
aturan_lama = ATURAN_UTAMA

try:
    ATURAN_UTAMA = {
        posisi: [] for posisi in PROFIL_POSISI_V3
    }

    _, detail_bukti_v3, _, review_v3, _ = rekomendasi_v2(
        baris_cv=baris_terpisah,
        referensi=referensi_bukti,
        kamus_lama=kamus_v3,
    )
finally:
    ATURAN_UTAMA = aturan_lama


def konteks_dasar(skill, teks):
    """
    Deteksi sederhana keterangan tingkat dasar dekat nama skill.
    Tidak menebak tingkat menengah/mahir jika tidak ada bukti.
    """
    alias = ALIAS_REKOM.get(
        skill, kamus_v3.get(skill, [skill])
    )

    penanda = r"(?:basics?|basic knowledge|beginner|dasar|pemula)"

    for nama in [skill, *alias]:
        token = re.escape(normal_rekom(nama))

        pola = (
            rf"(?<!\w){token}(?!\w)\s*[:()\-]?\s*{penanda}\b"
            rf"|\b{penanda}\s+(?:of\s+|in\s+)?{token}(?!\w)"
        )

        if re.search(pola, normal_rekom(teks)):
            return True

    return False


def kelompokkan_bukti(posisi, pilihan):
    semua = detail_bukti_v3.get(posisi, {}).get("bukti", [])

    temuan = [
        dict(b)
        for b in semua
        if b["kebutuhan"] in pilihan
    ]

    for b in temuan:
        b["konteks_tingkat"] = (
            "disebut dasar"
            if konteks_dasar(b["kebutuhan"], b["bukti"])
            else "tingkat tidak ditentukan"
        )

    return temuan


hasil_v3 = []
kelompok_v3 = []
bukti_tampil_v3 = []

for posisi, kelompok in PROFIL_POSISI_V3.items():
    jumlah_cocok = 0
    jumlah_proyek = 0
    utama_hilang = []
    dasar_saja = []

    for nama_kelompok, pilihan, jenis in kelompok:
        temuan = kelompokkan_bukti(posisi, pilihan)

        cocok = bool(temuan)
        dari_proyek = any(
            b["bagian"] in {"proyek", "pengalaman"}
            for b in temuan
        )

        hanya_dasar = bool(temuan) and all(
            b["konteks_tingkat"] == "disebut dasar"
            for b in temuan
        )

        jumlah_cocok += int(cocok)
        jumlah_proyek += int(dari_proyek)

        if jenis == "utama" and not cocok:
            utama_hilang.append(nama_kelompok)

        if hanya_dasar:
            dasar_saja.append(nama_kelompok)

        alternatif_ditemukan = sorted({
            b["kebutuhan"] for b in temuan
        })

        kelompok_v3.append({
            "posisi": posisi,
            "kelompok": nama_kelompok,
            "jenis": jenis,
            "alternatif": " / ".join(pilihan),
            "status": "ada bukti" if cocok else "belum ditemukan",
            "alternatif_ditemukan": ", ".join(alternatif_ditemukan),
            "bukti_proyek_pengalaman": dari_proyek,
            "hanya_keterangan_dasar": hanya_dasar,
        })

        for b in temuan:
            bukti_tampil_v3.append({
                "posisi": posisi,
                "kelompok": nama_kelompok,
                **b,
            })

    hasil_v3.append({
        "posisi": posisi,
        "kelompok_cocok": jumlah_cocok,
        "total_kelompok": len(kelompok),
        "cakupan_kelompok_persen": (
            100 * jumlah_cocok / len(kelompok)
        ),
        "kelompok_dengan_proyek_pengalaman": jumlah_proyek,
        "utama_belum_ditemukan": ", ".join(utama_hilang),
        "kelompok_hanya_dasar": ", ".join(dasar_saja),
        "lolos_bukti_utama": not utama_hilang,
    })


tabel_semua_v3 = pd.DataFrame(hasil_v3)
tabel_kelompok_v3 = pd.DataFrame(kelompok_v3)
tabel_bukti_v3 = pd.DataFrame(bukti_tampil_v3)

rekomendasi_v3_hasil = tabel_semua_v3.loc[
    tabel_semua_v3["lolos_bukti_utama"]
].copy()

rekomendasi_v3_hasil = rekomendasi_v3_hasil.sort_values(
    [
        "cakupan_kelompok_persen",
        "kelompok_dengan_proyek_pengalaman",
    ],
    ascending=[False, False],
    kind="stable",
).reset_index(drop=True)

if not rekomendasi_v3_hasil.empty:
    pasangan = list(zip(
        rekomendasi_v3_hasil["cakupan_kelompok_persen"],
        rekomendasi_v3_hasil["kelompok_dengan_proyek_pengalaman"],
    ))

    peringkat = {
        nilai: nomor
        for nomor, nilai in enumerate(dict.fromkeys(pasangan), 1)
    }

    rekomendasi_v3_hasil.insert(
        0, "peringkat",
        [peringkat[p] for p in pasangan],
    )

    rekomendasi_v3_hasil["cakupan_kelompok_persen"] = (
        rekomendasi_v3_hasil["cakupan_kelompok_persen"].round(2)
    )

print("CV:", globals().get("nama_cv", "CV aktif"))
print("VERSI ATURAN: kurasi-proyek-v3")
print("Cakupan: 12 posisi; bukan seluruh posisi dataset.")


if rekomendasi_v3_hasil.empty:
    print("\nBelum ada posisi yang memenuhi aturan bukti utama.")
else:
    # Tiga peringkat, sehingga posisi dengan nilai sama tidak
    # dipotong secara arbitrer.
    teratas = rekomendasi_v3_hasil.loc[
        rekomendasi_v3_hasil["peringkat"] <= 3
    ]

    print("\nREKOMENDASI AWAL")
    display(teratas[[
        "peringkat",
        "posisi",
        "cakupan_kelompok_persen",
        "kelompok_cocok",
        "total_kelompok",
        "kelompok_dengan_proyek_pengalaman",
        "kelompok_hanya_dasar",
    ]])

    for posisi in teratas["posisi"]:
        print(f"\n{posisi}")

        kelompok_posisi = tabel_kelompok_v3.loc[
            tabel_kelompok_v3["posisi"] == posisi
        ]

        display(kelompok_posisi.drop(columns="posisi"))

        if not tabel_bukti_v3.empty:
            display(
                tabel_bukti_v3.loc[
                    tabel_bukti_v3["posisi"] == posisi,
                    [
                        "kelompok",
                        "kebutuhan",
                        "baris_id",
                        "bagian",
                        "bukti",
                        "metode",
                        "konteks_tingkat",
                    ]
                ].drop_duplicates()
            )

        gap = kelompok_posisi.loc[
            kelompok_posisi["status"] == "belum ditemukan"
        ]

        if gap.empty:
            print(
                "Semua kelompok pada profil kurasi memiliki penyebutan "
                "atau bukti terkait. Tingkat penguasaan belum diverifikasi."
            )
        else:
            print("KEBUTUHAN YANG BELUM DITEMUKAN:")

            for _, g in gap.iterrows():
                print(
                    f"- {g['kelompok']} ({g['jenis']}): "
                    f"{g['alternatif']}."
                )

            print(
                "Jika sudah dikuasai, tambahkan bukti ke CV. "
                "Jika belum, pilih kebutuhan yang relevan dengan "
                "posisi tujuan. Skill dalam satu kelompok adalah alternatif."
            )


print("\nPOSISI DENGAN BUKTI UTAMA BELUM LENGKAP")
display(
    tabel_semua_v3.loc[
        ~tabel_semua_v3["lolos_bukti_utama"],
        ["posisi", "utama_belum_ditemukan", "kelompok_hanya_dasar"],
    ]
)

if review_v3:
    print("\nBUKTI DITAHAN UNTUK PEMERIKSAAN")
    display(pd.DataFrame(review_v3).drop_duplicates())

print(
    "\nCakupan kelompok adalah jumlah kelompok dengan bukti "
    "dibagi total kelompok dalam profil kurasi."
)
print(
    "Keterangan dasar hanya ditandai, belum diberi pengurang skor. "
    "Tidak ada keterangan tingkat bukan berarti mahir."
)
print(
    "Skor ini belum menggabungkan classifier, Sentence Transformer, "
    "durasi pengalaman, atau pendidikan."
)

CV: CV_Kesha_Amelia_Mobile_Developer.pdf
VERSI ATURAN: kurasi-proyek-v3
Cakupan: 12 posisi; bukan seluruh posisi dataset.

REKOMENDASI AWAL


,peringkat,posisi,cakupan_kelompok_persen,kelompok_cocok,total_kelompok,kelompok_dengan_proyek_pengalaman,kelompok_hanya_dasar
0,1,Frontend Developer,66.67,4,6,0,
1,2,React Developer,50.00,3,6,0,
2,3,Software Engineer,40.00,2,5,0,



Frontend Developer


,kelompok,jenis,alternatif,status,alternatif_ditemukan,bukti_proyek_pengalaman,hanya_keterangan_dasar
7,Bahasa frontend,utama,JavaScript / TypeScript,ada bukti,JavaScript,False,False
8,Framework frontend,utama,React / Vue.js / Angular,ada bukti,React,False,False
9,HTML,pendukung,HTML,belum ditemukan,,False,False
10,CSS,pendukung,CSS,belum ditemukan,,False,False
11,Integrasi API,pendukung,API Integration,ada bukti,API Integration,False,False
12,Version control,pendukung,Git,ada bukti,Git,False,False


,kelompok,kebutuhan,baris_id,bagian,bukti,metode,konteks_tingkat
6,Bahasa frontend,JavaScript,L0011,keterampilan,"Bahasa Pemrograman: Dart, Kotlin, Swift, JavaS...",penyebutan langsung,tingkat tidak ditentukan
7,Framework frontend,React,L0012,keterampilan,"Framework & Library: Flutter, React Native, Fi...",penyebutan langsung,tingkat tidak ditentukan
8,Integrasi API,API Integration,L0008,ringkasan,mengintegrasikan RESTful API dan layanan berba...,pola tindakan API; perlu review,tingkat tidak ditentukan
9,Version control,Git,L0014,keterampilan,"Tools & Cloud Platform: Git, CI/CD (GitHub Act...",penyebutan langsung,tingkat tidak ditentukan


...

,kelompok,jenis,alternatif,status,alternatif_ditemukan,bukti_proyek_pengalaman,hanya_keterangan_dasar
47,Bahasa pemrograman,utama,Python / Java / C++ / JavaScript,ada bukti,JavaScript,False,False
48,Version control,pendukung,Git,ada bukti,Git,False,False
49,Desain perangkat lunak,pendukung,Software Design,belum ditemukan,,False,False
50,Pengujian,pendukung,Test Automation / Manual Testing,belum ditemukan,,False,False
51,Problem solving,pendukung,Problem Solving,belum ditemukan,,False,False


,kelompok,kebutuhan,baris_id,bagian,bukti,metode,konteks_tingkat
30,Bahasa pemrograman,JavaScript,L0011,keterampilan,"Bahasa Pemrograman: Dart, Kotlin, Swift, JavaS...",penyebutan langsung,tingkat tidak ditentukan
31,Version control,Git,L0014,keterampilan,"Tools & Cloud Platform: Git, CI/CD (GitHub Act...",penyebutan langsung,tingkat tidak ditentukan


KEBUTUHAN YANG BELUM DITEMUKAN:
- Desain perangkat lunak (pendukung): Software Design.
- Pengujian (pendukung): Test Automation / Manual Testing.
- Problem solving (pendukung): Problem Solving.
Jika sudah dikuasai, tambahkan bukti ke CV. Jika belum, pilih kebutuhan yang relevan dengan posisi tujuan. Skill dalam satu kelompok adalah alternatif.

POSISI DENGAN BUKTI UTAMA BELUM LENGKAP


,posisi,utama_belum_ditemukan,kelompok_hanya_dasar
0,Full Stack Developer,Teknologi backend,
2,Backend Developer,"Bahasa/backend runtime, Pengembangan API",
3,Python Developer,Bahasa Python,
5,Vue Developer,Vue,
6,Angular Developer,"Angular, TypeScript",
7,Java Backend Developer,"Java, Pengembangan API",
9,DevOps Engineer,"Linux, Kontainer",
10,Data Scientist,"Bahasa analisis, Machine learning, Statistik",
11,Data Engineer,Pipeline data,



Cakupan kelompok adalah jumlah kelompok dengan bukti dibagi total kelompok dalam profil kurasi.
Keterangan dasar hanya ditandai, belum diberi pengurang skor. Tidak ada keterangan tingkat bukan berarti mahir.
Skor ini belum menggabungkan classifier, Sentence Transformer, durasi pengalaman, atau pendidikan.


In [24]:
# PERSIAPAN EVALUASI REKOMENDASI

import hashlib
import pandas as pd
from IPython.display import display

# Tidak menghapus hasil sebelumnya jika sel dijalankan ulang.
if "catatan_evaluasi" not in globals():
    catatan_evaluasi = []

print("Evaluasi siap.")
print("Jumlah CV yang sudah dicatat:", len(catatan_evaluasi))

Evaluasi siap.
Jumlah CV yang sudah dicatat: 0
